<a href="https://colab.research.google.com/github/L-Gaysina/eeg-tbi/blob/main/eeg_tbi_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Пайплайн предобработки ЭЭГ при ЧМТ и в контрольной группе

## Назначение ноутбука

Данный ноутбук реализует воспроизводимый пайплайн подготовки ЭЭГ-данных для сравнительного анализа пациентов с черепно-мозговой травмой (ЧМТ) и здоровых испытуемых контрольной группы.

В рамках ноутбука выполняются следующие этапы:

1. загрузка локальных файлов ЭЭГ;
2. приведение данных ЧМТ и контрольной группы к единому формату;
3. стандартизация частоты дискретизации, каналов и длительности эпох;
4. базовая предобработка сигнала;
5. контроль качества записей и эпох;
6. формирование итоговых таблиц признаков и служебных отчётов.

Ноутбук не выполняет скачивание данных из OpenNeuro. Предполагается, что контрольные EDF-файлы уже были заранее загружены в локальную папку, путь к которой задан в конфигурации.

> **Важно:** сырые ЭЭГ-данные не входят в репозиторий.  
> Ноутбук ожидает локальные пути к `.mat`-файлам ЧМТ и заранее скачанным EDF-файлам OpenNeuro `ds005385`.  
> Перед публикацией все реальные пути, идентификаторы файлов и выходы ячеек были очищены.

## Методическая схема анализа

Исследование построено как сравнительный анализ двух групп:

- **ЧМТ** — пациенты с черепно-мозговой травмой;
- **Контроль** — здоровые испытуемые из открытого датасета OpenNeuro ds005385.

Поскольку данные получены из разных источников и имеют разные исходные форматы, основной задачей данного этапа является их приведение к единому техническому стандарту.

Для этого используются следующие правила:

1. все записи переводятся в формат `mne.Epochs`;
2. анализ проводится на фиксированных 4-секундных эпохах;
3. частота дискретизации приводится к 500 Гц;
4. используется общий набор каналов, доступный в обеих группах;
5. данные фильтруются в диапазоне 0.5–70 Гц;
6. дополнительно применяется notch-фильтр для подавления сетевой помехи;
7. для всех записей используется единая схема референса;
8. результаты сохраняются в единую рабочую директорию `work/output`.

Такой подход позволяет снизить влияние технических различий между источниками данных и подготовить сопоставимое признаковое пространство для дальнейшего статистического анализа и машинного обучения.

## Входные данные

### Данные пациентов с ЧМТ

Данные пациентов с ЧМТ представлены в формате MATLAB `.mat` и содержат структуру EEGLAB `EEG`.

Ожидается, что в файлах могут присутствовать следующие поля:

- `EEG.data` — массив ЭЭГ-данных;
- `EEG.srate` — частота дискретизации;
- `EEG.chanlocs` — информация о каналах;
- `EEG.bad_chans` — предварительно размеченные плохие каналы, если доступны;
- `EEG.bad_epochs` — предварительно размеченные плохие эпохи, если доступны;
- ICA-поля, если независимые компоненты были рассчитаны ранее.

### Контрольные данные

Для контрольной группы используются EDF-файлы из датасета OpenNeuro ds005385.

В данном ноутбуке скачивание данных не выполняется. Предполагается, что EDF-файлы уже находятся в папке :
`CONFIG["ctl_ds005385_root"]`

Для анализа приоритетно используются записи состояния покоя с закрытыми глазами и протоколом acq-pre.


---

##  Ожидаемые результаты ноутбука


## Выходные данные

После выполнения ноутбука формируются следующие результаты:

1. стандартизированные ЭЭГ-записи в формате MNE;
2. таблица inventory с описанием доступных записей;
3. таблицы контроля качества;
4. таблицы признаков на уровне эпох, записей или субъектов;
5. служебные логи выполнения пайплайна;
6. файлы с ошибками, если часть записей не удалось обработать.

Все результаты сохраняются в рабочую директорию:


`work/output/`

Структура выходных данных задаётся автоматически на основе параметров конфигурации.

# Часть 1. Подготовка окружения

In [ ]:
# @title 0.1. Установка зависимостей

# В этой ячейке устанавливаются основные библиотеки,
# необходимые для обработки ЭЭГ, работы с таблицами и расчёта признаков.
#
# Важно:
# - предполагается, что EDF-файлы контрольной группы уже находятся
#   в локальной папке Google Drive.

!pip -q install mne scipy pandas numpy tqdm

In [ ]:
# @title 0.2. Импорт библиотек

# Стандартные библиотеки Python.
# Используются для работы с путями, логами, ошибками и служебными файлами.
import json
import logging
import math
import os
import re
import traceback
import warnings
from datetime import datetime
from pathlib import Path

# Библиотеки для научных вычислений и работы с таблицами.
import numpy as np
import pandas as pd
import scipy.signal as spsig

# Прогресс-бары для длинных циклов обработки записей.
from tqdm import tqdm

# MNE — основная библиотека для работы с ЭЭГ.
import mne
from mne.preprocessing import ICA

# Отключаем лишние предупреждения, чтобы вывод ноутбука был чище.
# При отладке эту строку можно временно закомментировать.
warnings.filterwarnings("ignore")

## 1. Подключение Google Drive (Colab)


## 1. Подключение Google Drive

На этом этапе подключается Google Drive, где хранятся:

- исходные данные пациентов с ЧМТ;
- заранее скачанные EDF-файлы контрольной группы;
- рабочая папка проекта;
- промежуточные и итоговые результаты обработки.

После подключения диска все пути задаются через объект `CONFIG`.

In [ ]:
# @title 1. Подключение Google Drive

# Google Drive используется как основное хранилище данных и результатов.
# После выполнения этой ячейки папки Drive будут доступны по пути:
# /content/drive/MyDrive/

from google.colab import drive

drive.mount("/content/drive")

# Часть 2. Конфигурация проекта



В этой ячейке задаются основные параметры пайплайна:

- пути к данным ЧМТ;
- путь к уже скачанным контрольным EDF-файлам;
- рабочая директория для результатов;
- параметры фильтрации и ресемплинга;
- настройки контроля качества;
- параметры ICA;
- режим тестового запуска.

Все ключевые параметры вынесены в словарь `CONFIG`, чтобы избежать дублирования путей и значений в разных частях ноутбука.


In [ ]:
# @title 2.1. Основная конфигурация проекта

# CONFIG — центральный словарь параметров.
# Все пути и основные настройки пайплайна задаются здесь.
#
# Такой подход делает ноутбук воспроизводимым:
# если структура папок изменилась, достаточно поправить только этот блок.

CONFIG = {
    # ------------------------------------------------------------------
    # Данные пациентов с ЧМТ
    # ------------------------------------------------------------------
    # Папка с .mat-файлами пациентов.
    "tbi_base_dir": Path("/path/to/local/tbi_mat_files"),

    # Шаблон поиска .mat-файлов внутри папки с данными ЧМТ.
    # "**/*.mat" означает поиск во всех вложенных папках.
    "tbi_mat_glob": "**/*.mat",

    # ------------------------------------------------------------------
    # Контрольная группа OpenNeuro ds005385
    # ------------------------------------------------------------------
    # Папка, в которой уже лежат скачанные EDF-файлы контрольной группы.
    # В этом ноутбуке скачивание не выполняется.
    "ctl_ds005385_root": Path("/path/to/local/openneuro/ds005385"),

    # Максимальное число контрольных файлов, которые будут выбраны
    # для дальнейшей обработки.
    #
    # Значение можно использовать для балансировки с числом записей ЧМТ.
    "ctl_target_n_files": None,

    # Сохранять ли стандартизированные эпохи в формате .fif.
    # Это ускоряет последующие этапы, но занимает место на диске.
    "save_standardized_epochs": True,

    # ------------------------------------------------------------------
    # Рабочая папка проекта
    # ------------------------------------------------------------------
    # Здесь будут сохраняться все результаты:
    # таблицы, логи, промежуточные файлы, отчёты качества.
    "work_dir": Path("/path/to/work_dir"),

    # ------------------------------------------------------------------
    # Тестовый режим
    # ------------------------------------------------------------------
    # Если True, пайплайн обрабатывает только небольшое число файлов.
    # Это удобно для отладки.
    "tbi_test_mode": False,
    "ctl_test_mode": False,

    # Стратегия выбора файлов в тестовом режиме:
    # - "first"  — первые N файлов;
    # - "random" — случайные N файлов;
    # - "slice" — срез по индексам;
    # - "list"  — явно заданный список файлов.
    "test_strategy": "first",

    # Число файлов для тестового запуска.
    "test_n_files": 2,

    # Seed нужен для воспроизводимого случайного выбора файлов.
    "test_seed": 97,

    # Срез файлов для режима "slice".
    "test_slice": (0, 5),

    # Явный список файлов для режима "list".
    "test_file_list": [],

    # ------------------------------------------------------------------
    # Стандартизация сигнала
    # ------------------------------------------------------------------
    # Целевая частота дискретизации.
    "target_sfreq": 500.0,

    # Нижняя и верхняя границы полосового фильтра.
    "l_freq": 0.5,
    "h_freq": 70.0,

    # Частоты notch-фильтра для подавления сетевой помехи и гармоник.
    "notch_freqs": [50, 100, 150, 200],

    # Тип референса.
    # "average" соответствует common average reference.
    "ref": "average",

    # ------------------------------------------------------------------
    # Контроль качества
    # ------------------------------------------------------------------
    # Использовать ли заранее размеченные плохие каналы/эпохи,
    # если они присутствуют в исходных данных.
    "use_premarked_bads": True,

    # Порог peak-to-peak амплитуды в микровольтах.
    # Эпохи с чрезмерно большой амплитудой могут быть исключены.
    "drop_epoch_p2p_uv": 350.0,
    # Минимальное число эпох для включения записи в анализ.
    "min_epochs_per_record": 10,

    # Максимально допустимая доля эпох выше p2p-порога.
    "max_bad_epoch_prop": 0.5,
    # ------------------------------------------------------------------

    # ------------------------------------------------------------------
    # ICA
    # ------------------------------------------------------------------
    # Выполнять ли ICA-очистку.
    "run_ica": True,

    # Метод ICA.
    # fastica используется как базовый устойчивый вариант.
    "ica_method": "fastica",

    # Число компонент ICA.
    # 0.99 означает сохранение компонент, объясняющих 99% дисперсии.
    "ica_n_components": 0.99,

    # Максимальное число итераций ICA.
    "ica_max_iter": 512,
}

In [ ]:
# @title 2.3.1 Единый стиль визуализаций

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Единые цвета групп во всём проекте.
COL_TBI = "#9B1B30"      # тёмно-красный для ЧМТ
COL_CONTROL = "#1F3F7A"  # тёмно-синий для контроля

GROUP_COLORS = {
    "TBI": COL_TBI,
    "Control": COL_CONTROL,
}

GROUP_LABELS_RU = {
    "TBI": "ЧМТ",
    "Control": "Контроль",
}

plt.rcParams.update(
    {
        "figure.figsize": (12, 7),
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.family": "serif",
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
    }
)


def save_figure(fig, output_path: Path) -> None:
    """
    Сохраняет рисунок в файл с аккуратными отступами.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Объект рисунка.
    output_path : Path
        Путь для сохранения изображения.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches="tight")
    print(f"Рисунок сохранён: {output_path}")

## 2.2. Структура выходных папок

На этом этапе создаётся единая структура папок для результатов пайплайна.

Используются отдельные директории для:

- логов выполнения;
- таблиц;
- промежуточных MNE-файлов;
- отчётов качества;
- ошибок обработки.

Это позволяет не смешивать исходные данные, промежуточные результаты и финальные таблицы.

In [ ]:
# @title 2.2. Создание структуры выходных папок

# Основная рабочая директория.
WORK_DIR = CONFIG["work_dir"]

# Единая папка для всех результатов.
OUTPUT_DIR = WORK_DIR / "output"

# Словарь с основными выходными директориями.
OUT = {
    "base": OUTPUT_DIR,
    "logs": OUTPUT_DIR / "logs",
    "tables": OUTPUT_DIR / "tables",
    "epochs": OUTPUT_DIR / "epochs",
    "qc": OUTPUT_DIR / "quality_control",
    "errors": OUTPUT_DIR / "errors",
}

# Создаём все папки, если они ещё не существуют.
for out_dir in OUT.values():
    out_dir.mkdir(parents=True, exist_ok=True)

print("Рабочая директория проекта:")
print(WORK_DIR)

print("\nПапка для результатов:")
print(OUTPUT_DIR)

print("\nСозданные выходные папки:")
for name, path in OUT.items():
    print(f"- {name}: {path}")

## 2.3. Проверка доступности входных данных

Перед запуском обработки необходимо проверить, доступны ли исходные файлы.

На этом этапе проверяется:

1. существует ли папка с `.mat`-файлами пациентов с ЧМТ;
2. существует ли папка с EDF-файлами контрольной группы;
3. сколько файлов найдено в каждой группе.

Если файлы не найдены, дальнейшая обработка не запускается.

In [ ]:
# @title Проверка путей к входным данным

def find_files(base_dir: Path, pattern: str) -> list[Path]:
    """
    Находит файлы по заданному шаблону.

    Parameters
    ----------
    base_dir : Path
        Папка, в которой выполняется поиск.
    pattern : str
        Шаблон поиска, например "**/*.mat" или "**/*.edf".

    Returns
    -------
    list[Path]
        Отсортированный список найденных файлов.
    """
    if not base_dir.exists():
        return []

    return sorted(base_dir.glob(pattern))


# Пути к данным.
tbi_base_dir = CONFIG["tbi_base_dir"]
ctl_base_dir = CONFIG["ctl_ds005385_root"]

# Поиск файлов.
tbi_mat_files = find_files(tbi_base_dir, CONFIG["tbi_mat_glob"])
ctl_edf_files = find_files(ctl_base_dir, "**/*.edf")

# Вывод краткой проверки.
print("Проверка входных данных")
print("-" * 40)

print(f"Папка ЧМТ существует: {tbi_base_dir.exists()}")
print(f"Найдено .mat-файлов ЧМТ: {len(tbi_mat_files)}")

print(f"\nПапка контроля существует: {ctl_base_dir.exists()}")
print(f"Найдено EDF-файлов контроля: {len(ctl_edf_files)}")

# Явная проверка, чтобы не продолжать выполнение с пустыми данными.
if len(tbi_mat_files) == 0:
    raise FileNotFoundError(
        "Не найдены .mat-файлы ЧМТ. "
        "Проверьте CONFIG['tbi_base_dir'] и CONFIG['tbi_mat_glob']."
    )

if len(ctl_edf_files) == 0:
    raise FileNotFoundError(
        "Не найдены EDF-файлы контрольной группы. "
        "Проверьте CONFIG['ctl_ds005385_root']."
    )

### Результат проверки данных

После выполнения предыдущей ячейки должны быть найдены:

- `.mat`-файлы пациентов с ЧМТ;
- `.edf`-файлы контрольной группы.

Если обе группы данных доступны, можно переходить к построению inventory — таблицы с описанием всех найденных записей.

Inventory используется для контроля состава выборки, выбора нужных протоколов и последующей проверки воспроизводимости анализа.

# 2.4. Логирование выполнения пайплайна


На этом этапе настраивается система логирования выполнения пайплайна.

Логирование используется для фиксации:

- времени запуска анализа;
- основных параметров конфигурации;
- количества найденных файлов;
- этапов обработки каждой записи;
- предупреждений и ошибок;
- путей к сохранённым промежуточным и итоговым файлам.

Это особенно важно при обработке большого числа ЭЭГ-записей, так как отдельные файлы могут иметь повреждённую структуру, неполный набор каналов или отличаться по техническим параметрам.

Ошибки отдельных файлов не должны останавливать весь пайплайн. Вместо этого они сохраняются в отдельную таблицу, чтобы их можно было проанализировать после завершения обработки.


In [ ]:
# @title  Настройка логирования

def setup_logger(log_dir: Path, logger_name: str = "eeg_pipeline") -> logging.Logger:
    """
    Создаёт и настраивает logger для пайплайна обработки ЭЭГ.

    Logger записывает сообщения одновременно:
    1. в консоль Colab;
    2. в текстовый лог-файл в папке output/logs.

    Parameters
    ----------
    log_dir : Path
        Папка для сохранения лог-файлов.
    logger_name : str, optional
        Имя logger, by default "eeg_pipeline".

    Returns
    -------
    logging.Logger
        Настроенный объект logger.
    """
    log_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = log_dir / f"{logger_name}_{timestamp}.log"

    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.INFO)

    # Если ячейка запускается повторно, старые handlers нужно удалить,
    # иначе сообщения будут дублироваться в выводе.
    if logger.handlers:
        logger.handlers.clear()

    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    console_handler = logging.StreamHandler()

    log_format = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler.setFormatter(log_format)
    console_handler.setFormatter(log_format)

    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    logger.info("Логирование пайплайна запущено.")
    logger.info("Лог-файл: %s", log_path)

    return logger


logger = setup_logger(OUT["logs"])

logger.info("Рабочая директория: %s", WORK_DIR)
logger.info("Папка результатов: %s", OUTPUT_DIR)
logger.info("Найдено .mat-файлов ЧМТ: %d", len(tbi_mat_files))
logger.info("Найдено EDF-файлов контроля: %d", len(ctl_edf_files))

## 2.5. Сохранение конфигурации запуска

Для обеспечения воспроизводимости текущая конфигурация пайплайна сохраняется в отдельный JSON-файл.

Это позволяет в дальнейшем восстановить параметры конкретного запуска анализа: пути к данным, настройки фильтрации, параметры ICA, режим тестового запуска и другие ключевые значения.

In [ ]:
# @title Сохранение конфигурации запуска

def convert_config_to_jsonable(config: dict) -> dict:
    """
    Преобразует словарь CONFIG в формат, пригодный для сохранения в JSON.

    Объекты Path и другие нестандартные типы преобразуются в строки.

    Parameters
    ----------
    config : dict
        Исходный словарь конфигурации.

    Returns
    -------
    dict
        Словарь, который можно сохранить в JSON.
    """
    jsonable_config = {}

    for key, value in config.items():
        if isinstance(value, Path):
            jsonable_config[key] = str(value)
        elif isinstance(value, tuple):
            jsonable_config[key] = list(value)
        else:
            jsonable_config[key] = value

    return jsonable_config


run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
config_path = OUT["logs"] / f"config_{run_timestamp}.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(
        convert_config_to_jsonable(CONFIG),
        file,
        ensure_ascii=False,
        indent=2,
    )

logger.info("Конфигурация запуска сохранена: %s", config_path)

print("Конфигурация запуска сохранена:")
print(config_path)

## 2.6. Служебная таблица ошибок

При обработке ЭЭГ отдельные файлы могут быть пропущены из-за ошибок чтения, неполного набора каналов, повреждённой структуры данных или несовместимого формата.

Для таких случаев создаётся отдельная таблица ошибок. В неё сохраняются:

- группа данных;
- путь к файлу;
- этап обработки, на котором возникла ошибка;
- тип ошибки;
- текст ошибки.

Такой подход позволяет не останавливать весь пайплайн из-за одного проблемного файла и затем отдельно проанализировать причины исключения записей.

In [ ]:
# @title  Служебные функции для регистрации ошибок

processing_errors = []


def register_processing_error(
    group: str,
    file_path: Path,
    stage: str,
    error: Exception,
) -> None:
    """
    Регистрирует ошибку обработки отдельного файла.

    Parameters
    ----------
    group : str
        Группа данных: "TBI" или "Control".
    file_path : Path
        Путь к файлу, при обработке которого возникла ошибка.
    stage : str
        Название этапа обработки.
    error : Exception
        Объект исключения.
    """
    error_record = {
        "group": group,
        "file_path": str(file_path),
        "stage": stage,
        "error_type": type(error).__name__,
        "error_message": str(error),
    }

    processing_errors.append(error_record)

    logger.error(
        "Ошибка обработки | group=%s | stage=%s | file=%s | error=%s",
        group,
        stage,
        file_path,
        str(error),
    )


def save_processing_errors(errors: list[dict], output_path: Path) -> None:
    """
    Сохраняет таблицу ошибок обработки.

    Parameters
    ----------
    errors : list[dict]
        Список зарегистрированных ошибок.
    output_path : Path
        Путь для сохранения CSV-файла.
    """
    if not errors:
        logger.info("Ошибки обработки не зарегистрированы.")
        return

    errors_df = pd.DataFrame(errors)
    errors_df.to_csv(output_path, index=False)

    logger.warning(
        "Таблица ошибок сохранена: %s | число ошибок: %d",
        output_path,
        len(errors_df),
    )

## 2.7. Безопасное выполнение операций

Для обработки большого числа файлов используется принцип безопасного выполнения: если ошибка возникает на одном файле, она регистрируется, но не останавливает весь пайплайн.

Это особенно важно для ЭЭГ-данных, так как отдельные записи могут отличаться по структуре, содержать неполный набор каналов или иметь технические проблемы при чтении.

In [ ]:
# @title Универсальная функция безопасного выполнения

def run_safely(
    func,
    group: str,
    file_path: Path,
    stage: str,
    default=None,
    **kwargs,
):
    """
    Выполняет функцию с перехватом ошибок.

    Если функция завершается с ошибкой, информация об ошибке
    сохраняется в processing_errors, а выполнение пайплайна продолжается.

    Parameters
    ----------
    func : callable
        Функция, которую нужно выполнить.
    group : str
        Группа данных: "TBI" или "Control".
    file_path : Path
        Путь к обрабатываемому файлу.
    stage : str
        Название этапа обработки.
    default : optional
        Значение, которое возвращается при ошибке.
    **kwargs
        Именованные аргументы, передаваемые в func.

    Returns
    -------
    object
        Результат выполнения func или default при ошибке.
    """
    try:
        return func(file_path=file_path, **kwargs)

    except Exception as error:
        register_processing_error(
            group=group,
            file_path=file_path,
            stage=stage,
            error=error,
        )
        return default

## 2.8. Проверка корректности конфигурации

Перед запуском обработки выполняется автоматическая проверка основных параметров конфигурации.

Проверяются:

- существование входных директорий;
- корректность частот фильтрации;
- корректность целевой частоты дискретизации;
- соответствие запрошенного числа контрольных файлов числу реально доступных EDF;
- параметры тестового режима.

Эта проверка позволяет обнаружить технические ошибки до начала длительной обработки данных.

In [ ]:
# @title Проверка корректности CONFIG

def validate_config(config: dict) -> None:
    """
    Проверяет корректность основных параметров CONFIG.

    Parameters
    ----------
    config : dict
        Словарь конфигурации пайплайна.

    Raises
    ------
    ValueError
        Если значения параметров некорректны.
    FileNotFoundError
        Если не найдены входные директории.
    """
    if not config["tbi_base_dir"].exists():
        raise FileNotFoundError(
            f"Не найдена папка с данными ЧМТ: {config['tbi_base_dir']}"
        )

    if not config["ctl_ds005385_root"].exists():
        raise FileNotFoundError(
            "Не найдена папка с контрольными EDF-файлами: "
            f"{config['ctl_ds005385_root']}"
        )

    if config["target_sfreq"] <= 0:
        raise ValueError("target_sfreq должна быть положительным числом.")

    if config["l_freq"] >= config["h_freq"]:
        raise ValueError("l_freq должна быть меньше h_freq.")

    if config["ctl_target_n_files"] <= 0:
        raise ValueError("ctl_target_n_files должно быть положительным числом.")

    if config["ctl_target_n_files"] > len(ctl_edf_files):
        logger.warning(
            "Запрошено %d контрольных файлов, но найдено только %d. "
            "Будут использованы все доступные EDF-файлы.",
            config["ctl_target_n_files"],
            len(ctl_edf_files),
        )

    if config["test_n_files"] <= 0:
        raise ValueError("test_n_files должно быть положительным числом.")

    allowed_test_strategies = {"first", "random", "slice", "list"}

    if config["test_strategy"] not in allowed_test_strategies:
        raise ValueError(
            "Недопустимое значение test_strategy. "
            f"Допустимые варианты: {allowed_test_strategies}"
        )


validate_config(CONFIG)

logger.info("Проверка CONFIG успешно завершена.")
print("CONFIG корректен. Можно переходить к построению inventory.")

### Итог служебной подготовки

На данном этапе:

- подключён Google Drive;
- задана конфигурация проекта;
- созданы выходные директории;
- проверена доступность входных данных;
- настроено логирование;
- сохранена конфигурация текущего запуска;
- подготовлены функции для регистрации ошибок.

Далее можно переходить к построению inventory — таблицы, описывающей все доступные записи ЧМТ и контрольной группы.

## Часть 3. Формирование inventory контрольной группы

На этом этапе формируется таблица `inventory` для контрольных EDF-файлов из OpenNeuro ds005385.

Inventory — это служебная таблица, в которой каждая строка соответствует одной найденной EDF-записи. Для каждой записи извлекаются:

- путь к файлу;
- идентификатор субъекта;
- идентификатор сессии;
- состояние записи (`EyesClosed` или `EyesOpen`);
- протокол регистрации (`acq-pre` или `acq-post`);
- имя файла.

Inventory необходим для воспроизводимого выбора контрольных записей и проверки того, какие именно файлы вошли в дальнейший анализ.

In [ ]:
# @title 3.1. Разбор имени EDF-файла OpenNeuro

def parse_openneuro_edf_path(file_path: Path) -> dict:
    """
    Извлекает метаданные из пути к EDF-файлу OpenNeuro.

    Ожидаемый формат имени файла:
    sub-001_ses-1_task-EyesClosed_acq-pre_eeg.edf

    Parameters
    ----------
    file_path : Path
        Путь к EDF-файлу.

    Returns
    -------
    dict
        Словарь с метаданными записи.
    """
    file_name = file_path.name

    subject_match = re.search(r"(sub-[A-Za-z0-9]+)", file_name)
    session_match = re.search(r"(ses-[A-Za-z0-9]+)", file_name)
    task_match = re.search(r"task-([A-Za-z0-9]+)", file_name)
    acq_match = re.search(r"acq-([A-Za-z0-9]+)", file_name)

    subject_id = subject_match.group(1) if subject_match else None
    session_id = session_match.group(1) if session_match else None
    task = task_match.group(1) if task_match else None
    acquisition = acq_match.group(1) if acq_match else None

    return {
        "group": "Control",
        "dataset": "ds005385",
        "subject_id": subject_id,
        "session_id": session_id,
        "task": task,
        "acquisition": acquisition,
        "file_name": file_name,
        "file_path": str(file_path),
    }

In [ ]:
# @title 3.2. Построение inventory для контрольных EDF-файлов

def build_control_inventory(edf_files: list[Path]) -> pd.DataFrame:
    """
    Формирует inventory-таблицу для контрольных EDF-файлов.

    Parameters
    ----------
    edf_files : list[Path]
        Список путей к EDF-файлам контрольной группы.

    Returns
    -------
    pd.DataFrame
        Таблица с метаданными найденных EDF-записей.
    """
    records = []

    for file_path in edf_files:
        record = parse_openneuro_edf_path(file_path)
        records.append(record)

    inventory = pd.DataFrame(records)

    sort_columns = [
        "subject_id",
        "session_id",
        "task",
        "acquisition",
        "file_name",
    ]

    inventory = inventory.sort_values(sort_columns).reset_index(drop=True)

    return inventory


control_inventory = build_control_inventory(ctl_edf_files)

control_inventory_path = OUT["tables"] / "control_inventory_ds005385_local.csv"
control_inventory.to_csv(control_inventory_path, index=False)

logger.info(
    "Inventory контрольной группы сохранён: %s | число записей: %d",
    control_inventory_path,
    len(control_inventory),
)

print("Inventory контрольной группы сформирован.")
print(f"Число EDF-записей: {len(control_inventory)}")
print(f"Число субъектов: {control_inventory['subject_id'].nunique()}")
print(f"Файл сохранён: {control_inventory_path}")

display(control_inventory.head())

## 3.3. Проверка состава контрольных протоколов

После построения inventory необходимо проверить, какие типы записей доступны в контрольной группе.

Для дальнейшего анализа приоритетным является состояние покоя с закрытыми глазами и протоколом `acq-pre`, так как оно лучше всего соответствует условиям регистрации данных ЧМТ.

Если таких записей недостаточно, могут использоваться дополнительные протоколы согласно заранее заданному приоритету.

## 3.4. Выбор контрольных записей для дальнейшей обработки

На этом этапе из всех доступных EDF-файлов контрольной группы выбирается подмножество записей для дальнейшей обработки.

Выбор выполняется по заранее заданному приоритету протоколов:

1. `EyesClosed`, `acq-pre`;
2. `EyesClosed`, `acq-post`;
3. `EyesOpen`, `acq-pre`;
4. `EyesOpen`, `acq-post`.

Такой подход позволяет сначала использовать наиболее релевантные записи, а затем при необходимости добрать контрольную выборку из менее приоритетных протоколов.

Количество выбранных файлов ограничивается параметром `CONFIG["ctl_target_n_files"]`.


In [ ]:
# @title Выбор контрольных EDF-файлов по приоритету протоколов

CONTROL_PROTOCOL_PRIORITY = [
    ("EyesClosed", "pre"),
    ("EyesClosed", "post"),
    ("EyesOpen", "pre"),
    ("EyesOpen", "post"),
]


def select_control_records(
    inventory: pd.DataFrame,
    protocol_priority: list[tuple[str, str]],
    target_n_files: int | None = None,
) -> pd.DataFrame:
    """
    Выбирает контрольные EDF-записи по приоритету протоколов.

    Если target_n_files=None, используются все записи,
    соответствующие заданным протоколам.

    Parameters
    ----------
    inventory : pd.DataFrame
        Inventory-таблица контрольных EDF-файлов.
    protocol_priority : list[tuple[str, str]]
        Список приоритетов в формате (task, acquisition).
    target_n_files : int | None, optional
        Максимальное число файлов для выбора. Если None, ограничение
        по числу файлов не применяется.

    Returns
    -------
    pd.DataFrame
        Таблица выбранных контрольных записей.
    """
    selected_parts = []

    for priority_rank, (task, acquisition) in enumerate(
        protocol_priority,
        start=1,
    ):
        protocol_records = inventory[
            (inventory["task"] == task)
            & (inventory["acquisition"] == acquisition)
        ].copy()

        protocol_records["protocol_priority"] = priority_rank
        selected_parts.append(protocol_records)

    selected = pd.concat(selected_parts, ignore_index=True)

    selected = selected.sort_values(
        [
            "protocol_priority",
            "subject_id",
            "session_id",
            "file_name",
        ]
    ).reset_index(drop=True)

    if target_n_files is not None and len(selected) > target_n_files:
        selected = selected.iloc[:target_n_files].copy()

    return selected.reset_index(drop=True)


control_selected = select_control_records(
    inventory=control_inventory,
    protocol_priority=CONTROL_PROTOCOL_PRIORITY,
    target_n_files=CONFIG["ctl_target_n_files"],
)

control_selected_path = OUT["tables"] / "control_selected_local_edf.csv"
control_selected.to_csv(control_selected_path, index=False)

logger.info(
    "Выбранные контрольные EDF сохранены: %s | число файлов: %d",
    control_selected_path,
    len(control_selected),
)

print("Выбор контрольных записей завершён.")
print(f"Выбрано EDF-файлов: {len(control_selected)}")
print(f"Число субъектов: {control_selected['subject_id'].nunique()}")
print(f"Файл сохранён: {control_selected_path}")

display(control_selected.head())

### Результат выбора контрольных записей

На данном этапе сформирована таблица выбранных EDF-файлов контрольной группы.

В таблице сохранены:

- идентификатор субъекта;
- сессия;
- состояние регистрации;
- протокол;
- путь к локальному EDF-файлу;
- приоритет протокола.

Эта таблица фиксирует состав контрольной выборки и используется на следующих этапах для загрузки, стандартизации и предобработки ЭЭГ.

Важно, что в данном ноутбуке не выполняется повторное скачивание данных: все выбранные файлы уже находятся в локальной директории проекта.

## Часть 4. Формирование inventory группы ЧМТ

На этом этапе формируется inventory-таблица для локальных `.mat`-файлов пациентов с черепно-мозговой травмой.

Данные пациентов с ЧМТ были предоставлены научным руководителем и получены на базе Института биофизики клетки РАН, г. Пущино. Данные были собраны квалифицированным врачом, обезличены и не распространяются в открытом доступе, так как являются закрытыми исследовательскими данными.

В данном ноутбуке не выполняется скачивание или внешняя загрузка данных ЧМТ. Используются только локальные `.mat`-файлы, путь к которым задан в параметре:

```python
CONFIG["tbi_base_dir"]

In [ ]:
# @title 4.1. Разбор пути к .mat-файлу ЧМТ

def parse_tbi_mat_path(file_path: Path) -> dict:
    """
    Извлекает базовые метаданные из пути к .mat-файлу ЧМТ.

    Поскольку данные ЧМТ являются локальными и могут иметь неоднородные
    имена файлов, функция использует устойчивую стратегию:

    1. record_id формируется из имени файла без расширения;
    2. subject_id пытается извлечься из имени файла;
    3. если subject_id не найден, используется имя родительской папки;
    4. если и это неинформативно, используется record_id.

    Parameters
    ----------
    file_path : Path
        Путь к .mat-файлу ЧМТ.

    Returns
    -------
    dict
        Словарь с базовыми метаданными записи.
    """
    file_stem = file_path.stem
    parent_name = file_path.parent.name

    subject_patterns = [
        r"(sub[-_ ]?\d+)",
        r"(subject[-_ ]?\d+)",
        r"(patient[-_ ]?\d+)",
        r"(pt[-_ ]?\d+)",
        r"(^\d+)",
    ]

    subject_id = None

    for pattern in subject_patterns:
        match = re.search(pattern, file_stem, flags=re.IGNORECASE)
        if match:
            subject_id = match.group(1)
            break

    if subject_id is None and parent_name:
        subject_id = parent_name

    if subject_id is None:
        subject_id = file_stem

    return {
        "group": "TBI",
        "dataset": "local_tbi",
        "subject_id": subject_id,
        "record_id": file_stem,
        "file_name": file_path.name,
        "parent_dir": parent_name,
        "file_size_mb": file_path.stat().st_size / (1024 ** 2),
        "file_path": str(file_path),
    }

In [ ]:
# @title 4.2. Построение inventory для .mat-файлов ЧМТ

def build_tbi_inventory(mat_files: list[Path]) -> pd.DataFrame:
    """
    Формирует inventory-таблицу для локальных .mat-файлов ЧМТ.

    Parameters
    ----------
    mat_files : list[Path]
        Список путей к .mat-файлам ЧМТ.

    Returns
    -------
    pd.DataFrame
        Таблица с базовыми метаданными записей ЧМТ.
    """
    records = []

    for file_path in mat_files:
        record = parse_tbi_mat_path(file_path)
        records.append(record)

    inventory = pd.DataFrame(records)

    inventory = inventory.sort_values(
        ["subject_id", "record_id", "file_name"]
    ).reset_index(drop=True)

    return inventory


tbi_inventory = build_tbi_inventory(tbi_mat_files)

tbi_inventory_path = OUT["tables"] / "tbi_inventory_local_mat.csv"
tbi_inventory.to_csv(tbi_inventory_path, index=False)

logger.info(
    "Inventory группы ЧМТ сохранён: %s | число записей: %d",
    tbi_inventory_path,
    len(tbi_inventory),
)

print("Inventory группы ЧМТ сформирован.")
print(f"Число .mat-файлов: {len(tbi_inventory)}")
print(f"Число субъектов: {tbi_inventory['subject_id'].nunique()}")
print(f"Файл сохранён: {tbi_inventory_path}")

display(tbi_inventory.head())

## 4.3. Проверка состава группы ЧМТ

После построения inventory проверяется соответствие найденных файлов ожидаемой структуре выборки.

В исходном наборе данных ЧМТ ожидается:

- 215 записей;
- 92 субъекта.

Если фактическое число субъектов отличается от ожидаемого, это может означать, что идентификатор субъекта некорректно извлекается из имени файла или структуры папок. В таком случае необходимо уточнить правило формирования `subject_id`.

In [ ]:
# @title Контрольная проверка состава группы ЧМТ

expected_tbi_records = 215
expected_tbi_subjects = 92

actual_tbi_records = len(tbi_inventory)
actual_tbi_subjects = tbi_inventory["subject_id"].nunique()

print("Контрольная проверка состава группы ЧМТ")
print("-" * 50)
print(f"Ожидаемое число записей: {expected_tbi_records}")
print(f"Фактическое число записей: {actual_tbi_records}")
print(f"Ожидаемое число субъектов: {expected_tbi_subjects}")
print(f"Фактическое число субъектов: {actual_tbi_subjects}")

if actual_tbi_records != expected_tbi_records:
    logger.warning(
        "Число записей ЧМТ отличается от ожидаемого: %d вместо %d",
        actual_tbi_records,
        expected_tbi_records,
    )

if actual_tbi_subjects != expected_tbi_subjects:
    logger.warning(
        "Число субъектов ЧМТ отличается от ожидаемого: %d вместо %d",
        actual_tbi_subjects,
        expected_tbi_subjects,
    )

if (
    actual_tbi_records == expected_tbi_records
    and actual_tbi_subjects == expected_tbi_subjects
):
    print("\nСостав группы ЧМТ соответствует ожидаемому.")

In [ ]:
# @title 4.4. Сводка по числу записей на субъекта в группе ЧМТ

tbi_subject_summary = (
    tbi_inventory
    .groupby("subject_id")
    .agg(
        n_records=("record_id", "count"),
        total_size_mb=("file_size_mb", "sum"),
        median_file_size_mb=("file_size_mb", "median"),
    )
    .reset_index()
    .sort_values(["n_records", "subject_id"], ascending=[False, True])
)

tbi_subject_summary_path = OUT["tables"] / "tbi_subject_summary.csv"
tbi_subject_summary.to_csv(tbi_subject_summary_path, index=False)

print("Сводка по числу записей на субъекта в группе ЧМТ:")
display(tbi_subject_summary.head(20))

print("\nОписание распределения числа записей на субъекта:")
display(tbi_subject_summary["n_records"].describe())

## 4.5. Объединённая сводка состава выборки

После построения отдельных inventory для ЧМТ и контрольной группы формируется общая сводка состава данных.

Она используется для быстрой проверки числа субъектов и записей в каждой группе перед запуском предобработки.

In [ ]:
# @title Объединённая сводка состава выборки

sample_summary = pd.DataFrame(
    [
        {
            "group": "TBI",
            "group_ru": "ЧМТ",
            "n_subjects": tbi_inventory["subject_id"].nunique(),
            "n_records": len(tbi_inventory),
        },
        {
            "group": "Control",
            "group_ru": "Контроль",
            "n_subjects": control_selected["subject_id"].nunique(),
            "n_records": len(control_selected),
        },
    ]
)

sample_summary_path = OUT["tables"] / "sample_summary_inventory.csv"
sample_summary.to_csv(sample_summary_path, index=False)

print("Сводка состава выборки:")
display(sample_summary)



In [ ]:
# @title 4.6. Контрольная проверка ожидаемого состава выборки

expected_sample_summary = {
    "TBI": {
        "n_subjects": 92,
        "n_records": 215,
    },
    "Control": {
        "n_subjects": 72,
        "n_records": 364,
    },
}

actual_sample_summary = {
    row["group"]: {
        "n_subjects": int(row["n_subjects"]),
        "n_records": int(row["n_records"]),
    }
    for _, row in sample_summary.iterrows()
}

print("Контрольная проверка состава выборки")
print("-" * 50)

for group, expected_values in expected_sample_summary.items():
    actual_values = actual_sample_summary[group]

    print(f"\nГруппа: {group}")
    print(
        "Субъекты: "
        f"{actual_values['n_subjects']} / ожидалось {expected_values['n_subjects']}"
    )
    print(
        "Записи: "
        f"{actual_values['n_records']} / ожидалось {expected_values['n_records']}"
    )

    if actual_values != expected_values:
        logger.warning(
            "Состав группы %s отличается от ожидаемого: %s вместо %s",
            group,
            actual_values,
            expected_values,
        )

if actual_sample_summary == expected_sample_summary:
    print("\nСостав обеих групп соответствует ожидаемому.")

## 4.7. Визуальная сводка состава выборки

Для быстрой проверки состава данных строится обзорный график числа субъектов и записей в каждой группе.

Этот рисунок не является результатом анализа ЭЭГ, а используется как техническая визуализация состава выборки перед запуском предобработки.

In [ ]:
# @title 4.7. Визуальная сводка состава выборки

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_data = sample_summary.copy()
plot_data["color"] = plot_data["group"].map(GROUP_COLORS)
plot_data["group_ru"] = plot_data["group"].map(GROUP_LABELS_RU)

axes[0].bar(
    plot_data["group_ru"],
    plot_data["n_subjects"],
    color=plot_data["color"],
)
axes[0].set_title("Число субъектов")
axes[0].set_ylabel("Количество субъектов")

for index, row in plot_data.iterrows():
    axes[0].text(
        index,
        row["n_subjects"] + 1,
        str(row["n_subjects"]),
        ha="center",
        va="bottom",
    )

axes[1].bar(
    plot_data["group_ru"],
    plot_data["n_records"],
    color=plot_data["color"],
)
axes[1].set_title("Число записей")
axes[1].set_ylabel("Количество записей")

for index, row in plot_data.iterrows():
    axes[1].text(
        index,
        row["n_records"] + 5,
        str(row["n_records"]),
        ha="center",
        va="bottom",
    )

fig.suptitle("Состав исходной выборки", fontsize=16, y=1.03)
fig.tight_layout()

figure_path = OUT["qc"] / "sample_composition_overview.png"
save_figure(fig, figure_path)

plt.show()

### Итог inventory-этапа

В результате inventory-этапа были сформированы таблицы для обеих групп:

- `tbi_inventory_local_mat.csv` — локальные `.mat`-файлы пациентов с ЧМТ;
- `control_inventory_ds005385_local.csv` — все найденные EDF-файлы контрольной группы OpenNeuro ds005385;
- `control_selected_local_edf.csv` — EDF-файлы, передаваемые в дальнейшую обработку;
- `sample_summary_inventory.csv` — общая сводка числа субъектов и записей.

Итоговый состав данных перед предобработкой:

| Группа | Число субъектов | Число записей |
|---|---:|---:|
| ЧМТ | 92 | 215 |
| Контроль | 72 | 364 |

После проверки состава выборки можно переходить к загрузке ЭЭГ и приведению записей к единому формату.

# Часть 5. Загрузка и стандартизация ЭЭГ

После формирования inventory начинается основной этап подготовки сигналов.

Цель этого этапа — привести записи ЧМТ и контрольной группы к единому формату, пригодному для последующего расчёта признаков.

Поскольку исходные данные имеют разные форматы:

- ЧМТ: локальные `.mat`-файлы со структурой EEGLAB;
- Контроль: EDF-файлы OpenNeuro ds005385;

необходимо выполнить унифицированную процедуру загрузки и стандартизации.

На этом этапе для каждой записи выполняются следующие операции:

1. чтение исходного файла;
2. преобразование данных в объект MNE;
3. проверка частоты дискретизации;
4. приведение частоты дискретизации к 500 Гц;
5. унификация названий каналов;
6. фильтрация сигнала;
7. применение notch-фильтра;
8. установка common average reference;
9. сегментация или проверка 4-секундных эпох;
10. сохранение стандартизированных данных и служебной информации.

Итогом этапа является набор сопоставимых ЭЭГ-записей, подготовленных к дальнейшему контролю качества и расчёту признаков.

## 5.1. Общие параметры стандартизации

Для обеих групп используются единые параметры обработки сигнала.

Ключевые настройки берутся из `CONFIG`:

- целевая частота дискретизации — 500 Гц;
- полосовой фильтр — 0.5–70 Гц;
- notch-фильтр — 50 Гц и гармоники;
- референс — common average reference;
- длительность эпохи — 4 секунды.

Единые параметры предобработки необходимы для того, чтобы минимизировать влияние технических различий между источниками данных.

In [ ]:
# @title Общие параметры стандартизации ЭЭГ

TARGET_SFREQ = CONFIG["target_sfreq"]
L_FREQ = CONFIG["l_freq"]
H_FREQ = CONFIG["h_freq"]
NOTCH_FREQS = CONFIG["notch_freqs"]
REFERENCE = CONFIG["ref"]

# Длительность эпохи в секундах.
EPOCH_LENGTH_SEC = 4.0

print("Параметры стандартизации ЭЭГ")
print("-" * 50)
print(f"Целевая частота дискретизации: {TARGET_SFREQ} Гц")
print(f"Полосовой фильтр: {L_FREQ}–{H_FREQ} Гц")
print(f"Notch-фильтр: {NOTCH_FREQS}")
print(f"Референс: {REFERENCE}")
print(f"Длительность эпохи: {EPOCH_LENGTH_SEC} с")

## 5.2. Унификация названий каналов

Перед объединением данных из разных источников необходимо привести названия каналов к единому виду.

В разных форматах один и тот же канал может быть записан по-разному, например:

- `FP1`;
- `Fp1`;
- `EEG Fp1`;
- `Fp1-Ref`.

Для дальнейшего анализа такие названия должны быть стандартизированы. Это необходимо для корректного пересечения каналов между группами, интерполяции плохих каналов и расчёта признаков по ROI.

In [ ]:
# @title Функции унификации названий каналов

def clean_channel_name(channel_name: str) -> str:
    """
    Приводит название ЭЭГ-канала к единому формату.

    Parameters
    ----------
    channel_name : str
        Исходное название канала.

    Returns
    -------
    str
        Стандартизированное название канала.
    """
    cleaned_name = str(channel_name).strip()

    # Удаляем распространённые служебные префиксы и суффиксы.
    cleaned_name = re.sub(r"^EEG\s+", "", cleaned_name, flags=re.IGNORECASE)
    cleaned_name = re.sub(r"-Ref$", "", cleaned_name, flags=re.IGNORECASE)
    cleaned_name = re.sub(r"-REF$", "", cleaned_name, flags=re.IGNORECASE)

    # Убираем лишние пробелы.
    cleaned_name = cleaned_name.replace(" ", "")

    # Приводим типичные названия 10–20 к привычному регистру.
    replacements = {
        "FP1": "Fp1",
        "FP2": "Fp2",
        "FZ": "Fz",
        "CZ": "Cz",
        "PZ": "Pz",
        "OZ": "Oz",
    }

    upper_name = cleaned_name.upper()

    if upper_name in replacements:
        return replacements[upper_name]

    return cleaned_name


def standardize_channel_names(raw: mne.io.BaseRaw) -> mne.io.BaseRaw:
    """
    Стандартизирует названия каналов в объекте MNE Raw.

    Parameters
    ----------
    raw : mne.io.BaseRaw
        Объект Raw с ЭЭГ-сигналом.

    Returns
    -------
    mne.io.BaseRaw
        Raw-объект с обновлёнными названиями каналов.
    """
    rename_mapping = {
        channel_name: clean_channel_name(channel_name)
        for channel_name in raw.ch_names
    }

    raw = raw.copy()
    raw.rename_channels(rename_mapping)

    return raw

## 5.3. Базовая предобработка Raw-сигнала

После загрузки запись приводится к единому виду.

Выполняются следующие операции:

1. стандартизация названий каналов;
2. выбор только ЭЭГ-каналов;
3. установка стандартной схемы расположения электродов;
4. ресемплинг до целевой частоты дискретизации;
5. полосовая фильтрация;
6. notch-фильтрация;
7. применение common average reference.

Эта функция не выполняет ICA и расширенный контроль качества. Эти этапы оформляются отдельно, чтобы предобработка оставалась прозрачной и воспроизводимой.

In [ ]:
# @title Базовая предобработка Raw-сигнала

def preprocess_raw_basic(
    raw: mne.io.BaseRaw,
    target_sfreq: float,
    l_freq: float,
    h_freq: float,
    notch_freqs: list[float],
    reference: str = "average",
) -> mne.io.BaseRaw:
    """
    Выполняет базовую стандартизацию Raw-сигнала.

    Parameters
    ----------
    raw : mne.io.BaseRaw
        Исходный Raw-объект MNE.
    target_sfreq : float
        Целевая частота дискретизации.
    l_freq : float
        Нижняя граница полосового фильтра.
    h_freq : float
        Верхняя граница полосового фильтра.
    notch_freqs : list[float]
        Частоты notch-фильтра.
    reference : str, optional
        Тип референса, by default "average".

    Returns
    -------
    mne.io.BaseRaw
        Предобработанный Raw-объект.
    """
    raw = raw.copy()

    # Приводим названия каналов к единому формату.
    raw = standardize_channel_names(raw)

    # Оставляем только ЭЭГ-каналы, если в записи есть служебные каналы.
    raw.pick_types(eeg=True, exclude=[])

    # Устанавливаем стандартную 10–20/10–10 монтажную схему.
    # on_missing="ignore" позволяет не прерывать обработку,
    # если для части каналов нет координат.
    montage = mne.channels.make_standard_montage("standard_1020")
    raw.set_montage(montage, on_missing="ignore")

    # Ресемплинг выполняется только если частота отличается от целевой.
    current_sfreq = raw.info["sfreq"]

    if not np.isclose(current_sfreq, target_sfreq):
        raw.resample(target_sfreq)

    # Полосовая фильтрация.
    raw.filter(
        l_freq=l_freq,
        h_freq=h_freq,
        fir_design="firwin",
        verbose=False,
    )

    # Notch-фильтр для подавления сетевой помехи.
    available_notch_freqs = [
        freq for freq in notch_freqs
        if freq < raw.info["sfreq"] / 2
    ]

    if available_notch_freqs:
        raw.notch_filter(
            freqs=available_notch_freqs,
            verbose=False,
        )

    # Common average reference.
    if reference == "average":
        raw.set_eeg_reference("average", projection=False, verbose=False)

    return raw

## 5.4. Сегментация на фиксированные эпохи

Для сопоставимости записей обеих групп данные разбиваются на фиксированные 4-секундные эпохи.

Такой формат удобен для последующего расчёта спектральных, временных, событийных и сетевых признаков.

Сегментация выполняется без перекрытия. Каждая эпоха рассматривается как отдельный фрагмент сигнала, но итоговые признаки далее агрегируются на уровне записи или субъекта.

In [ ]:
# @title Сегментация Raw на 4-секундные эпохи

def make_fixed_length_epochs(
    raw: mne.io.BaseRaw,
    epoch_length_sec: float = 4.0,
) -> mne.Epochs:
    """
    Разбивает Raw-сигнал на фиксированные эпохи.

    Parameters
    ----------
    raw : mne.io.BaseRaw
        Предобработанный Raw-объект.
    epoch_length_sec : float, optional
        Длительность эпохи в секундах, by default 4.0.

    Returns
    -------
    mne.Epochs
        Объект Epochs с фиксированными неперекрывающимися эпохами.
    """
    events = mne.make_fixed_length_events(
        raw,
        id=1,
        duration=epoch_length_sec,
    )

    epochs = mne.Epochs(
        raw,
        events,
        event_id={"fixed": 1},
        tmin=0.0,
        tmax=epoch_length_sec - 1.0 / raw.info["sfreq"],
        baseline=None,
        preload=True,
        verbose=False,
    )

    return epochs

## 5.5. Извлечение служебной информации о записи

После загрузки и стандартизации каждой записи сохраняется краткая техническая информация:

- группа;
- идентификатор субъекта;
- идентификатор записи;
- число каналов;
- частота дискретизации;
- число эпох;
- длительность эпохи;
- список каналов.

Эта информация используется для контроля качества и проверки сопоставимости данных между группами.

In [ ]:
# @title Извлечение служебной информации о записи

def summarize_epochs(
    epochs: mne.Epochs,
    group: str,
    subject_id: str,
    record_id: str,
    source_path: str,
) -> dict:
    """
    Формирует краткую техническую сводку по объекту Epochs.

    Parameters
    ----------
    epochs : mne.Epochs
        Объект эпох MNE.
    group : str
        Группа данных.
    subject_id : str
        Идентификатор субъекта.
    record_id : str
        Идентификатор записи.
    source_path : str
        Путь к исходному файлу.

    Returns
    -------
    dict
        Словарь с технической информацией о записи.
    """
    sfreq = float(epochs.info["sfreq"])
    n_epochs = len(epochs)
    n_channels = len(epochs.ch_names)

    epoch_length_sec = (
        epochs.tmax - epochs.tmin + 1.0 / sfreq
    )

    return {
        "group": group,
        "subject_id": subject_id,
        "record_id": record_id,
        "source_path": source_path,
        "n_epochs": n_epochs,
        "n_channels": n_channels,
        "sfreq_hz": sfreq,
        "epoch_len_sec": epoch_length_sec,
        "tmin_sec": epochs.tmin,
        "tmax_sec": epochs.tmax,
        "channels": "|".join(epochs.ch_names),
    }

### Итог подготовки общих функций

На данном этапе определены общие функции, которые будут использоваться для обеих групп:

- стандартизация названий каналов;
- базовая предобработка Raw-сигнала;
- сегментация на 4-секундные эпохи;
- извлечение технической информации о записи.

Далее эти функции применяются отдельно к данным ЧМТ и контрольной группы, поскольку исходные форматы файлов различаются.

## 5.6. Загрузка и стандартизация контрольных EDF-файлов

На этом этапе выполняется загрузка EDF-файлов контрольной группы OpenNeuro ds005385.

Каждая запись обрабатывается по единой схеме:

1. чтение локального EDF-файла;
2. преобразование в объект `mne.io.Raw`;
3. стандартизация названий каналов;
4. выбор ЭЭГ-каналов;
5. установка стандартного монтажа;
6. ресемплинг до 500 Гц при необходимости;
7. полосовая фильтрация 0.5–70 Гц;
8. notch-фильтрация;
9. применение common average reference;
10. разбиение на фиксированные 4-секундные эпохи;
11. сохранение технической информации о записи.

В данном ноутбуке скачивание EDF-файлов не выполняется. Все файлы читаются из локальной папки, указанной в `CONFIG["ctl_ds005385_root"]`.

In [ ]:
# @title 5.6.1. Загрузка одной контрольной EDF-записи

def load_control_edf_as_epochs(
    file_path: Path,
    target_sfreq: float,
    l_freq: float,
    h_freq: float,
    notch_freqs: list[float],
    reference: str = "average",
    epoch_length_sec: float = 4.0,
) -> mne.Epochs:
    """
    Загружает EDF-файл контрольной группы и преобразует его в MNE Epochs.

    Parameters
    ----------
    file_path : Path
        Путь к локальному EDF-файлу.
    target_sfreq : float
        Целевая частота дискретизации.
    l_freq : float
        Нижняя граница полосового фильтра.
    h_freq : float
        Верхняя граница полосового фильтра.
    notch_freqs : list[float]
        Частоты notch-фильтра.
    reference : str, optional
        Тип референса, by default "average".
    epoch_length_sec : float, optional
        Длительность фиксированной эпохи, by default 4.0.

    Returns
    -------
    mne.Epochs
        Стандартизированные 4-секундные эпохи.
    """
    raw = mne.io.read_raw_edf(
        file_path,
        preload=True,
        verbose=False,
    )

    raw = preprocess_raw_basic(
        raw=raw,
        target_sfreq=target_sfreq,
        l_freq=l_freq,
        h_freq=h_freq,
        notch_freqs=notch_freqs,
        reference=reference,
    )

    epochs = make_fixed_length_epochs(
        raw=raw,
        epoch_length_sec=epoch_length_sec,
    )

    return epochs

## 5.7. Тестовая загрузка одной контрольной записи

Перед запуском обработки всех EDF-файлов выполняется тестовая загрузка одной записи.

Это позволяет проверить:

- корректность пути к файлу;
- возможность чтения EDF через MNE;
- частоту дискретизации после стандартизации;
- число каналов;
- число полученных 4-секундных эпох;
- отсутствие ошибок на базовом этапе предобработки.

Если тестовая загрузка завершается успешно, можно переходить к массовой обработке контрольной группы.

In [ ]:
# @title Тестовая загрузка одной контрольной записи

test_control_row = control_selected.iloc[0]
test_control_path = Path(test_control_row["file_path"])

print("Тестовая контрольная запись")
print("-" * 50)
print(f"Файл: {test_control_path}")
print(f"Субъект: {test_control_row['subject_id']}")
print(f"Протокол: {test_control_row['task']} / {test_control_row['acquisition']}")

test_control_epochs = load_control_edf_as_epochs(
    file_path=test_control_path,
    target_sfreq=TARGET_SFREQ,
    l_freq=L_FREQ,
    h_freq=H_FREQ,
    notch_freqs=NOTCH_FREQS,
    reference=REFERENCE,
    epoch_length_sec=EPOCH_LENGTH_SEC,
)

test_control_summary = summarize_epochs(
    epochs=test_control_epochs,
    group="Control",
    subject_id=test_control_row["subject_id"],
    record_id=test_control_row["file_name"].replace(".edf", ""),
    source_path=str(test_control_path),
)

print("\nРезультат тестовой загрузки:")
for key, value in test_control_summary.items():
    if key != "channels":
        print(f"{key}: {value}")

print("\nПервые 10 каналов:")
print(test_control_epochs.ch_names[:10])

## 5.8. Массовая обработка контрольной группы

На этом этапе единая процедура загрузки и стандартизации применяется ко всем выбранным EDF-файлам контрольной группы.

Для каждой записи сохраняется техническая информация:

- идентификатор субъекта;
- идентификатор записи;
- число эпох;
- число каналов;
- частота дискретизации;
- длительность эпохи;
- список каналов;
- путь к исходному файлу;
- путь к сохранённому MNE-файлу, если сохранение включено.

Ошибки отдельных файлов регистрируются в служебной таблице и не останавливают выполнение всего пайплайна.

In [ ]:
# @title Массовая обработка контрольных EDF-файлов

def process_control_record(
    row: pd.Series,
    output_dir: Path,
    save_epochs: bool = True,
) -> dict:
    """
    Обрабатывает одну контрольную EDF-запись.

    Parameters
    ----------
    row : pd.Series
        Строка из control_selected.
    output_dir : Path
        Папка для сохранения стандартизированных эпох.
    save_epochs : bool, optional
        Сохранять ли epochs в .fif, by default True.

    Returns
    -------
    dict
        Техническая сводка по обработанной записи.
    """
    file_path = Path(row["file_path"])
    subject_id = row["subject_id"]
    record_id = file_path.stem

    epochs = load_control_edf_as_epochs(
        file_path=file_path,
        target_sfreq=TARGET_SFREQ,
        l_freq=L_FREQ,
        h_freq=H_FREQ,
        notch_freqs=NOTCH_FREQS,
        reference=REFERENCE,
        epoch_length_sec=EPOCH_LENGTH_SEC,
    )

    summary = summarize_epochs(
        epochs=epochs,
        group="Control",
        subject_id=subject_id,
        record_id=record_id,
        source_path=str(file_path),
    )

    summary["task"] = row.get("task")
    summary["acquisition"] = row.get("acquisition")
    summary["protocol_priority"] = row.get("protocol_priority")

    if save_epochs:
        group_output_dir = output_dir / "control"
        group_output_dir.mkdir(parents=True, exist_ok=True)

        epochs_path = group_output_dir / f"{record_id}_epo.fif"

        epochs.save(
            epochs_path,
            overwrite=True,
            verbose=False,
        )

        summary["epochs_path"] = str(epochs_path)
    else:
        summary["epochs_path"] = None

    return summary


control_epoch_summaries = []

save_standardized_epochs = CONFIG.get("save_standardized_epochs", True)

for _, row in tqdm(
    control_selected.iterrows(),
    total=len(control_selected),
    desc="Обработка контроля",
):
    file_path = Path(row["file_path"])

    try:
        summary = process_control_record(
            row=row,
            output_dir=OUT["epochs"],
            save_epochs=save_standardized_epochs,
        )
        control_epoch_summaries.append(summary)

    except Exception as error:
        register_processing_error(
            group="Control",
            file_path=file_path,
            stage="control_edf_standardization",
            error=error,
        )

control_epochs_inventory = pd.DataFrame(control_epoch_summaries)

control_epochs_inventory_path = (
    OUT["tables"] / "control_epochs_inventory_standardized.csv"
)

control_epochs_inventory.to_csv(
    control_epochs_inventory_path,
    index=False,
)

logger.info(
    "Inventory стандартизированных контрольных эпох сохранён: %s | записей: %d",
    control_epochs_inventory_path,
    len(control_epochs_inventory),
)

print("Обработка контрольной группы завершена.")
print(f"Успешно обработано записей: {len(control_epochs_inventory)}")
print(f"Ошибок обработки: {len(processing_errors)}")
print(f"Файл сохранён: {control_epochs_inventory_path}")

display(control_epochs_inventory.head())

## 5.9. Проверка результата обработки контрольной группы

После массовой обработки контрольной группы необходимо проверить технические характеристики полученных эпох.

Проверяются:

- число успешно обработанных записей;
- число субъектов;
- частота дискретизации;
- длительность эпохи;
- число каналов;
- распределение числа эпох на запись.

Эта проверка нужна для раннего выявления несоответствий до перехода к обработке группы ЧМТ.

In [ ]:
# @title Проверка стандартизированных контрольных эпох

print("Проверка контрольной группы после стандартизации")
print("-" * 60)

print(f"Успешно обработано записей: {len(control_epochs_inventory)}")
print(
    "Число субъектов: "
    f"{control_epochs_inventory['subject_id'].nunique()}"
)

print("\nЧастоты дискретизации:")
display(control_epochs_inventory["sfreq_hz"].value_counts().sort_index())

print("\nДлительность эпох:")
display(
    control_epochs_inventory["epoch_len_sec"]
    .round(6)
    .value_counts()
    .sort_index()
)

print("\nЧисло каналов:")
display(control_epochs_inventory["n_channels"].value_counts().sort_index())

print("\nОписание числа эпох на запись:")
display(control_epochs_inventory["n_epochs"].describe())

## 5.10. Загрузка и стандартизация данных ЧМТ

На этом этапе выполняется загрузка локальных `.mat`-файлов пациентов с черепно-мозговой травмой.

Исходные данные ЧМТ представлены в формате MATLAB/EEGLAB и содержат структуру `EEG`. В зависимости от файла структура может включать:

- `EEG.data` — массив ЭЭГ-сигнала;
- `EEG.srate` — частоту дискретизации;
- `EEG.chanlocs` — информацию о каналах;
- `EEG.bad_chans` — предварительно размеченные плохие каналы, если они доступны;
- `EEG.bad_epochs` — предварительно размеченные плохие эпохи, если они доступны.

Основная задача этого этапа — преобразовать данные ЧМТ в объект `mne.Epochs`, сопоставимый с контрольной группой.

Для каждой записи выполняются следующие операции:

1. чтение локального `.mat`-файла;
2. извлечение структуры `EEG`;
3. извлечение массива данных, частоты дискретизации и названий каналов;
4. преобразование данных в формат MNE;
5. стандартизация названий каналов;
6. ресемплинг до 500 Гц при необходимости;
7. фильтрация 0.5–70 Гц;
8. notch-фильтрация;
9. применение common average reference;
10. сохранение технической информации о записи.

In [ ]:
# @title 5.10.1. Чтение .mat-файла ЧМТ и извлечение структуры EEG

def load_mat_file(file_path: Path) -> dict:
    """
    Загружает MATLAB .mat-файл.

    Parameters
    ----------
    file_path : Path
        Путь к .mat-файлу.

    Returns
    -------
    dict
        Содержимое .mat-файла.
    """
    import scipy.io

    mat_data = scipy.io.loadmat(
        file_path,
        squeeze_me=True,
        struct_as_record=False,
    )

    return mat_data


def extract_eeg_struct(mat_data: dict):
    """
    Извлекает структуру EEG из загруженного .mat-файла.

    Parameters
    ----------
    mat_data : dict
        Содержимое .mat-файла.

    Returns
    -------
    object
        Структура EEG.

    Raises
    ------
    KeyError
        Если структура EEG не найдена.
    """
    if "EEG" in mat_data:
        return mat_data["EEG"]

    # Иногда ключ может иметь другой регистр.
    for key in mat_data.keys():
        if key.lower() == "eeg":
            return mat_data[key]

    raise KeyError("В .mat-файле не найдена структура EEG.")

### Извлечение основных полей EEGLAB

Из структуры `EEG` извлекаются три ключевых элемента:

- массив сигнала;
- частота дискретизации;
- названия каналов.

Для MNE данные должны иметь форму:

```text
n_epochs × n_channels × n_times

In [ ]:
# @title 5.10.2. Извлечение данных, частоты и каналов из EEGLAB

def get_eeg_field(eeg_struct, field_name: str):
    """
    Извлекает поле из структуры EEG.

    Parameters
    ----------
    eeg_struct : object
        Структура EEG.
    field_name : str
        Название поля.

    Returns
    -------
    object
        Значение поля.

    Raises
    ------
    AttributeError
        Если поле отсутствует.
    """
    if hasattr(eeg_struct, field_name):
        return getattr(eeg_struct, field_name)

    raise AttributeError(f"В структуре EEG отсутствует поле: {field_name}")


def extract_channel_names_from_chanlocs(chanlocs) -> list[str]:
    """
    Извлекает названия каналов из EEG.chanlocs.

    Parameters
    ----------
    chanlocs : object
        Поле EEG.chanlocs.

    Returns
    -------
    list[str]
        Список названий каналов.
    """
    channel_names = []

    if np.ndim(chanlocs) == 0:
        chanlocs = [chanlocs]

    for channel in np.ravel(chanlocs):
        if hasattr(channel, "labels"):
            channel_names.append(str(channel.labels))
        elif isinstance(channel, dict) and "labels" in channel:
            channel_names.append(str(channel["labels"]))
        else:
            channel_names.append(f"EEG{len(channel_names) + 1}")

    return channel_names


def ensure_epochs_shape(data: np.ndarray, n_channels: int) -> np.ndarray:
    """
    Приводит массив ЭЭГ к форме n_epochs × n_channels × n_times.

    Parameters
    ----------
    data : np.ndarray
        Исходный массив ЭЭГ.
    n_channels : int
        Ожидаемое число каналов.

    Returns
    -------
    np.ndarray
        Массив формы n_epochs × n_channels × n_times.

    Raises
    ------
    ValueError
        Если форму массива невозможно интерпретировать.
    """
    data = np.asarray(data)

    if data.ndim == 3:
        # Возможные варианты:
        # 1) n_channels × n_times × n_epochs
        # 2) n_epochs × n_channels × n_times
        # 3) n_channels × n_epochs × n_times

        if data.shape[0] == n_channels:
            data = np.transpose(data, (2, 0, 1))
        elif data.shape[1] == n_channels:
            data = data
        elif data.shape[0] != n_channels and data.shape[1] != n_channels:
            raise ValueError(
                "Не удалось определить ось каналов в 3D-массиве EEG.data."
            )

    elif data.ndim == 2:
        # Если данные непрерывные: n_channels × n_times.
        # Преобразуем к одной эпохе.
        if data.shape[0] == n_channels:
            data = data[np.newaxis, :, :]
        elif data.shape[1] == n_channels:
            data = data.T[np.newaxis, :, :]
        else:
            raise ValueError(
                "Не удалось определить ось каналов в 2D-массиве EEG.data."
            )

    else:
        raise ValueError(
            f"Неподдерживаемая размерность EEG.data: {data.ndim}"
        )

    return data

In [ ]:
# @title 5.10.3. Преобразование EEGLAB-структуры в MNE Epochs

def convert_eeglab_struct_to_epochs(eeg_struct) -> mne.Epochs:
    """
    Преобразует структуру EEGLAB EEG в объект MNE Epochs.

    Parameters
    ----------
    eeg_struct : object
        Структура EEG из .mat-файла.

    Returns
    -------
    mne.Epochs
        Объект MNE Epochs.
    """
    data = get_eeg_field(eeg_struct, "data")
    sfreq = float(get_eeg_field(eeg_struct, "srate"))
    chanlocs = get_eeg_field(eeg_struct, "chanlocs")

    channel_names = extract_channel_names_from_chanlocs(chanlocs)
    channel_names = [clean_channel_name(name) for name in channel_names]

    data = ensure_epochs_shape(
        data=np.asarray(data),
        n_channels=len(channel_names),
    )

    # MNE ожидает данные в вольтах.
    # Если значения похожи на микровольты, переводим в вольты.
    median_abs_value = np.nanmedian(np.abs(data))

    if median_abs_value > 1e-3:
        data = data * 1e-6

    info = mne.create_info(
        ch_names=channel_names,
        sfreq=sfreq,
        ch_types="eeg",
    )

    epochs = mne.EpochsArray(
        data=data,
        info=info,
        tmin=0.0,
        baseline=None,
        verbose=False,
    )

    montage = mne.channels.make_standard_montage("standard_1020")
    epochs.set_montage(montage, on_missing="ignore")

    return epochs

### Базовая стандартизация уже эпохированных данных

Данные ЧМТ уже представлены в виде фиксированных фрагментов, поэтому после чтения `.mat` они преобразуются сразу в `mne.Epochs`.

К объекту `Epochs` применяются те же параметры стандартизации, что и к контрольной группе:

- ресемплинг до 500 Гц;
- фильтрация 0.5–70 Гц;
- notch-фильтрация;
- common average reference;
- унификация названий каналов.

In [ ]:
# @title 5.10.4. Базовая предобработка Epochs-сигнала

def preprocess_epochs_basic(
    epochs: mne.Epochs,
    target_sfreq: float,
    l_freq: float,
    h_freq: float,
    notch_freqs: list[float],
    reference: str = "average",
) -> mne.Epochs:
    """
    Выполняет базовую стандартизацию объекта MNE Epochs.

    Parameters
    ----------
    epochs : mne.Epochs
        Исходный объект эпох MNE.
    target_sfreq : float
        Целевая частота дискретизации.
    l_freq : float
        Нижняя граница полосового фильтра.
    h_freq : float
        Верхняя граница полосового фильтра.
    notch_freqs : list[float]
        Частоты notch-фильтра.
    reference : str, optional
        Тип референса, by default "average".

    Returns
    -------
    mne.Epochs
        Предобработанный объект Epochs.
    """
    epochs = epochs.copy()

    rename_mapping = {
        channel_name: clean_channel_name(channel_name)
        for channel_name in epochs.ch_names
    }
    epochs.rename_channels(rename_mapping)

    # Современный вариант выбора только ЭЭГ-каналов.
    epochs.pick("eeg")

    montage = mne.channels.make_standard_montage("standard_1020")
    epochs.set_montage(montage, on_missing="ignore")

    current_sfreq = epochs.info["sfreq"]

    if not np.isclose(current_sfreq, target_sfreq):
        epochs.resample(target_sfreq)

    epochs.filter(
        l_freq=l_freq,
        h_freq=h_freq,
        fir_design="firwin",
        verbose=False,
    )

    available_notch_freqs = [
        freq for freq in notch_freqs
        if freq < epochs.info["sfreq"] / 2
    ]

    if available_notch_freqs:
        data = epochs.get_data(copy=True)

        filtered_data = mne.filter.notch_filter(
            x=data,
            Fs=epochs.info["sfreq"],
            freqs=available_notch_freqs,
            verbose=False,
        )

        epochs = mne.EpochsArray(
            data=filtered_data,
            info=epochs.info.copy(),
            events=epochs.events.copy(),
            event_id=epochs.event_id.copy(),
            tmin=epochs.tmin,
            baseline=None,
            metadata=epochs.metadata,
            verbose=False,
        )

    if reference == "average":
        epochs.set_eeg_reference("average", projection=False, verbose=False)

    return epochs

Для данных ЧМТ notch-фильтрация применяется к массиву эпох через `mne.filter.notch_filter`, поскольку объект `EpochsArray` в используемой версии MNE не поддерживает метод `.notch_filter()` напрямую. После фильтрации объект `EpochsArray` пересобирается с сохранением метаданных, событий и информации о каналах.

In [ ]:
# @title 5.10.5. Загрузка одной .mat-записи ЧМТ

def load_tbi_mat_as_epochs(
    file_path: Path,
    target_sfreq: float,
    l_freq: float,
    h_freq: float,
    notch_freqs: list[float],
    reference: str = "average",
) -> mne.Epochs:
    """
    Загружает .mat-файл ЧМТ и преобразует его в MNE Epochs.

    Parameters
    ----------
    file_path : Path
        Путь к локальному .mat-файлу.
    target_sfreq : float
        Целевая частота дискретизации.
    l_freq : float
        Нижняя граница полосового фильтра.
    h_freq : float
        Верхняя граница полосового фильтра.
    notch_freqs : list[float]
        Частоты notch-фильтра.
    reference : str, optional
        Тип референса, by default "average".

    Returns
    -------
    mne.Epochs
        Стандартизированные эпохи ЭЭГ.
    """
    mat_data = load_mat_file(file_path)
    eeg_struct = extract_eeg_struct(mat_data)

    epochs = convert_eeglab_struct_to_epochs(eeg_struct)

    epochs = preprocess_epochs_basic(
        epochs=epochs,
        target_sfreq=target_sfreq,
        l_freq=l_freq,
        h_freq=h_freq,
        notch_freqs=notch_freqs,
        reference=reference,
    )

    return epochs

## 5.11. Тестовая загрузка одной записи ЧМТ

Перед массовой обработкой группы ЧМТ выполняется тестовая загрузка одной `.mat`-записи.

Это позволяет проверить:

- корректность чтения MATLAB-файла;
- наличие структуры `EEG`;
- форму массива данных;
- корректность извлечения каналов;
- частоту дискретизации;
- длительность эпох;
- совместимость данных с объектом `mne.Epochs`.

Если тестовая загрузка проходит успешно, можно переходить к обработке всех записей группы ЧМТ.

In [ ]:
# @title Тестовая загрузка одной записи ЧМТ

test_tbi_row = tbi_inventory.iloc[0]
test_tbi_path = Path(test_tbi_row["file_path"])

print("Тестовая запись ЧМТ")
print("-" * 50)
print(f"Файл: {test_tbi_path}")
print(f"Субъект: {test_tbi_row['subject_id']}")
print(f"Запись: {test_tbi_row['record_id']}")

test_tbi_epochs = load_tbi_mat_as_epochs(
    file_path=test_tbi_path,
    target_sfreq=TARGET_SFREQ,
    l_freq=L_FREQ,
    h_freq=H_FREQ,
    notch_freqs=NOTCH_FREQS,
    reference=REFERENCE,
)

test_tbi_summary = summarize_epochs(
    epochs=test_tbi_epochs,
    group="TBI",
    subject_id=test_tbi_row["subject_id"],
    record_id=test_tbi_row["record_id"],
    source_path=str(test_tbi_path),
)

print("\nРезультат тестовой загрузки:")
for key, value in test_tbi_summary.items():
    if key != "channels":
        print(f"{key}: {value}")

print("\nПервые 10 каналов:")
print(test_tbi_epochs.ch_names[:10])

## 5.12. Массовая обработка группы ЧМТ

На этом этапе процедура загрузки и стандартизации применяется ко всем `.mat`-файлам группы ЧМТ.

Для каждой записи сохраняется техническая информация:

- идентификатор субъекта;
- идентификатор записи;
- число эпох;
- число каналов;
- частота дискретизации;
- длительность эпохи;
- список каналов;
- путь к исходному `.mat`-файлу;
- путь к сохранённому MNE-файлу, если сохранение включено.

Если отдельный файл не удаётся обработать, ошибка регистрируется в служебной таблице, но выполнение пайплайна продолжается.

In [ ]:
# @title Массовая обработка .mat-файлов ЧМТ

def process_tbi_record(
    row: pd.Series,
    output_dir: Path,
    save_epochs: bool = True,
) -> dict:
    """
    Обрабатывает одну .mat-запись ЧМТ.

    Parameters
    ----------
    row : pd.Series
        Строка из tbi_inventory.
    output_dir : Path
        Папка для сохранения стандартизированных эпох.
    save_epochs : bool, optional
        Сохранять ли epochs в .fif, by default True.

    Returns
    -------
    dict
        Техническая сводка по обработанной записи.
    """
    file_path = Path(row["file_path"])
    subject_id = row["subject_id"]
    record_id = row["record_id"]

    epochs = load_tbi_mat_as_epochs(
        file_path=file_path,
        target_sfreq=TARGET_SFREQ,
        l_freq=L_FREQ,
        h_freq=H_FREQ,
        notch_freqs=NOTCH_FREQS,
        reference=REFERENCE,
    )

    summary = summarize_epochs(
        epochs=epochs,
        group="TBI",
        subject_id=subject_id,
        record_id=record_id,
        source_path=str(file_path),
    )

    if save_epochs:
        group_output_dir = output_dir / "tbi"
        group_output_dir.mkdir(parents=True, exist_ok=True)

        safe_record_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(record_id))
        epochs_path = group_output_dir / f"{safe_record_id}_epo.fif"

        epochs.save(
            epochs_path,
            overwrite=True,
            verbose=False,
        )

        summary["epochs_path"] = str(epochs_path)
    else:
        summary["epochs_path"] = None

    return summary


tbi_epoch_summaries = []
save_standardized_epochs = CONFIG.get("save_standardized_epochs", True)

for _, row in tqdm(
    tbi_inventory.iterrows(),
    total=len(tbi_inventory),
    desc="Обработка ЧМТ",
):
    file_path = Path(row["file_path"])

    try:
        summary = process_tbi_record(
            row=row,
            output_dir=OUT["epochs"],
            save_epochs=save_standardized_epochs,
        )
        tbi_epoch_summaries.append(summary)

    except Exception as error:
        register_processing_error(
            group="TBI",
            file_path=file_path,
            stage="tbi_mat_standardization",
            error=error,
        )

tbi_epochs_inventory = pd.DataFrame(tbi_epoch_summaries)

tbi_epochs_inventory_path = (
    OUT["tables"] / "tbi_epochs_inventory_standardized.csv"
)

tbi_epochs_inventory.to_csv(
    tbi_epochs_inventory_path,
    index=False,
)

logger.info(
    "Inventory стандартизированных эпох ЧМТ сохранён: %s | записей: %d",
    len(tbi_epochs_inventory),
)

print("Обработка группы ЧМТ завершена.")
print(f"Успешно обработано записей: {len(tbi_epochs_inventory)}")
print(f"Ошибок обработки: {len(processing_errors)}")
print(f"Файл сохранён: {tbi_epochs_inventory_path}")

display(tbi_epochs_inventory.head())

Обработка ЧМТ:   5%|▌         | 11/215 [02:37<49:52, 14.67s/it]Exception ignored in: <function ZipFile.__del__ at 0x7dd7a8c77ec0>
Traceback (most recent call last):
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1966, in __del__
    self.close()
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1975, in close
    raise ValueError("Can't close the ZIP file while there is "
ValueError: Can't close the ZIP file while there is an open writing handle on it. Close the writing handle before closing the zip.
Обработка ЧМТ:  20%|██        | 44/215 [10:53<41:58, 14.73s/it]

## 5.13. Проверка результата обработки группы ЧМТ

После массовой обработки группы ЧМТ необходимо проверить технические характеристики полученных эпох.

Проверяются:

- число успешно обработанных записей;
- число субъектов;
- частота дискретизации;
- длительность эпохи;
- число каналов;
- распределение числа эпох на запись.

Эта проверка позволяет убедиться, что данные ЧМТ были приведены к формату, сопоставимому с контрольной группой.

In [ ]:
# @title Проверка стандартизированных эпох ЧМТ

print("Проверка группы ЧМТ после стандартизации")
print("-" * 60)

print(f"Успешно обработано записей: {len(tbi_epochs_inventory)}")
print(
    "Число субъектов: "
    f"{tbi_epochs_inventory['subject_id'].nunique()}"
)

print("\nЧастоты дискретизации:")
display(tbi_epochs_inventory["sfreq_hz"].value_counts().sort_index())

print("\nДлительность эпох:")
display(
    tbi_epochs_inventory["epoch_len_sec"]
    .round(6)
    .value_counts()
    .sort_index()
)

print("\nЧисло каналов:")
display(tbi_epochs_inventory["n_channels"].value_counts().sort_index())

print("\nОписание числа эпох на запись:")
display(tbi_epochs_inventory["n_epochs"].describe())

## 5.14. Объединённая проверка стандартизированных данных

После отдельной обработки ЧМТ и контрольной группы формируется общая таблица стандартизированных записей.

Эта таблица используется для проверки сопоставимости групп перед дальнейшим контролем качества и расчётом признаков.

На этом этапе проверяются:

- число успешно обработанных записей в каждой группе;
- число субъектов;
- частота дискретизации;
- длительность эпох;
- число каналов;
- число эпох на запись.

In [ ]:
# @title Объединение inventory стандартизированных эпох

standardized_inventory = pd.concat(
    [
        tbi_epochs_inventory,
        control_epochs_inventory,
    ],
    ignore_index=True,
)

standardized_inventory_path = (
    OUT["tables"] / "inventory_epochs_standardized.csv"
)

standardized_inventory.to_csv(
    standardized_inventory_path,
    index=False,
)

print("Объединённый inventory стандартизированных эпох:")
print(f"Всего записей: {len(standardized_inventory)}")
print(f"Файл сохранён: {standardized_inventory_path}")

display(
    standardized_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
        median_epochs_per_record=("n_epochs", "median"),
        median_channels=("n_channels", "median"),
        median_sfreq=("sfreq_hz", "median"),
        median_epoch_len_sec=("epoch_len_sec", "median"),
    )
    .reset_index()
)

In [ ]:
# @title 5.14.1. Визуальная сводка стандартизированных данных

plot_df = standardized_inventory.copy()

plot_df["group_ru"] = plot_df["group"].map(GROUP_LABELS_RU)
plot_df["color"] = plot_df["group"].map(GROUP_COLORS)

summary_for_plot = (
    plot_df
    .groupby(["group", "group_ru"])
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
        median_epochs=("n_epochs", "median"),
        median_channels=("n_channels", "median"),
        median_sfreq=("sfreq_hz", "median"),
        median_epoch_len=("epoch_len_sec", "median"),
    )
    .reset_index()
)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Число записей.
axes[0, 0].bar(
    summary_for_plot["group_ru"],
    summary_for_plot["n_records"],
    color=summary_for_plot["group"].map(GROUP_COLORS),
)
axes[0, 0].set_title("Число стандартизированных записей")
axes[0, 0].set_ylabel("Количество записей")

for index, row in summary_for_plot.iterrows():
    axes[0, 0].text(
        index,
        row["n_records"] + 5,
        str(int(row["n_records"])),
        ha="center",
        va="bottom",
    )

# 2. Общее число эпох.
axes[0, 1].bar(
    summary_for_plot["group_ru"],
    summary_for_plot["n_epochs"],
    color=summary_for_plot["group"].map(GROUP_COLORS),
)
axes[0, 1].set_title("Общее число 4-секундных эпох")
axes[0, 1].set_ylabel("Количество эпох")

for index, row in summary_for_plot.iterrows():
    axes[0, 1].text(
        index,
        row["n_epochs"] + 300,
        str(int(row["n_epochs"])),
        ha="center",
        va="bottom",
    )

# 3. Медианное число эпох на запись.
axes[1, 0].bar(
    summary_for_plot["group_ru"],
    summary_for_plot["median_epochs"],
    color=summary_for_plot["group"].map(GROUP_COLORS),
)
axes[1, 0].set_title("Медианное число эпох на запись")
axes[1, 0].set_ylabel("Медиана числа эпох")

for index, row in summary_for_plot.iterrows():
    axes[1, 0].text(
        index,
        row["median_epochs"] + 2,
        f"{row['median_epochs']:.0f}",
        ha="center",
        va="bottom",
    )

# 4. Медианное число каналов.
axes[1, 1].bar(
    summary_for_plot["group_ru"],
    summary_for_plot["median_channels"],
    color=summary_for_plot["group"].map(GROUP_COLORS),
)
axes[1, 1].set_title("Медианное число каналов до унификации")
axes[1, 1].set_ylabel("Количество каналов")

for index, row in summary_for_plot.iterrows():
    axes[1, 1].text(
        index,
        row["median_channels"] + 1,
        f"{row['median_channels']:.0f}",
        ha="center",
        va="bottom",
    )

fig.suptitle(
    "Сводка стандартизированных ЭЭГ-записей",
    fontsize=18,
    y=1.02,
)

fig.tight_layout()

figure_path = OUT["qc"] / "standardized_inventory_overview.png"
save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 5.15. Проверка технической сопоставимости групп

technical_summary = (
    standardized_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs_total=("n_epochs", "sum"),
        sfreq_values=("sfreq_hz", lambda x: sorted(x.unique())),
        n_channels_values=("n_channels", lambda x: sorted(x.unique())),
        epoch_len_values=(
            "epoch_len_sec",
            lambda x: sorted(np.round(x.unique(), 6)),
        ),
    )
    .reset_index()
)

technical_summary_path = OUT["tables"] / "technical_summary_standardized.csv"
technical_summary.to_csv(technical_summary_path, index=False)

display(technical_summary)

logger.info(
    "Техническая сводка стандартизированных данных сохранена: %s",
    technical_summary_path,
)

### Итог этапа загрузки и стандартизации

В результате этапа загрузки и стандартизации данные ЧМТ и контрольной группы были приведены к единому формату MNE Epochs.

Итоговый стандартизированный набор данных включает:

- записи ЧМТ, загруженные из локальных `.mat`-файлов;
- контрольные записи, загруженные из локальных EDF-файлов OpenNeuro ds005385;
- единую частоту дискретизации;
- фиксированную длительность эпох;
- унифицированные названия каналов;
- сохранённую техническую информацию по каждой записи.

Следующий этап — унификация набора каналов между группами и контроль качества записей.


# 6. Унификация набора каналов между группами

После загрузки и базовой стандартизации данных необходимо привести записи ЧМТ и контрольной группы к единому набору ЭЭГ-каналов.

Это необходимо по нескольким причинам:

1. данные ЧМТ и контрольной группы получены из разных источников;
2. исходные названия и состав каналов могут отличаться;
3. статистическое сравнение и машинное обучение требуют единого признакового пространства;
4. признаки по ROI должны рассчитываться на сопоставимых электродах;
5. сетевые метрики функциональной связности требуют одинакового набора узлов.

На этом этапе определяется пересечение каналов, доступных в обеих группах, после чего все записи приводятся к этому общему набору.



## 6.1. Проверка набора каналов после стандартизации

На этом этапе анализируется состав каналов в стандартизированных записях.

Для каждой записи ранее был сохранён список каналов в поле `channels`. Теперь эти списки используются для проверки:

- сколько каналов содержит каждая запись;
- какие каналы встречаются в группе ЧМТ;
- какие каналы встречаются в контрольной группе;
- какие каналы являются общими для обеих групп.

In [ ]:
# @title 6.1. Проверка списков каналов после стандартизации

def split_channel_string(channel_string: str) -> list[str]:
    """
    Преобразует строку с каналами в список названий каналов.

    Parameters
    ----------
    channel_string : str
        Строка с каналами, разделёнными символом "|".

    Returns
    -------
    list[str]
        Список названий каналов.
    """
    if pd.isna(channel_string):
        return []

    return [
        channel.strip()
        for channel in str(channel_string).split("|")
        if channel.strip()
    ]


def get_group_channel_sets(
    inventory: pd.DataFrame,
    group_name: str,
) -> list[set[str]]:
    """
    Возвращает список множеств каналов для записей одной группы.

    Parameters
    ----------
    inventory : pd.DataFrame
        Inventory стандартизированных записей.
    group_name : str
        Название группы.

    Returns
    -------
    list[set[str]]
        Список множеств каналов по записям.
    """
    group_inventory = inventory[inventory["group"] == group_name]

    channel_sets = [
        set(split_channel_string(channel_string))
        for channel_string in group_inventory["channels"]
    ]

    return channel_sets


tbi_channel_sets = get_group_channel_sets(
    inventory=standardized_inventory,
    group_name="TBI",
)

control_channel_sets = get_group_channel_sets(
    inventory=standardized_inventory,
    group_name="Control",
)

print("Проверка каналов после стандартизации")
print("-" * 60)
print(f"Число записей ЧМТ: {len(tbi_channel_sets)}")
print(f"Число записей контроля: {len(control_channel_sets)}")

print("\nЧисло каналов в записях ЧМТ:")
display(
    standardized_inventory
    .query("group == 'TBI'")
    ["n_channels"]
    .value_counts()
    .sort_index()
)

print("\nЧисло каналов в записях контроля:")
display(
    standardized_inventory
    .query("group == 'Control'")
    ["n_channels"]
    .value_counts()
    .sort_index()
)

## 6.2. Определение общего набора каналов

Для дальнейшего анализа используется только общий набор каналов, присутствующий во всех стандартизированных записях обеих групп.

Такой подход позволяет избежать ситуации, когда признаки в разных группах рассчитываются по разным электродам.

Общий набор каналов определяется как пересечение каналов по всем успешно обработанным записям ЧМТ и контрольной группы.

In [ ]:
# @title 6.2. Определение общего набора каналов

def intersect_channel_sets(channel_sets: list[set[str]]) -> set[str]:
    """
    Находит пересечение каналов по списку множеств.

    Parameters
    ----------
    channel_sets : list[set[str]]
        Список множеств каналов.

    Returns
    -------
    set[str]
        Пересечение каналов.
    """
    if not channel_sets:
        return set()

    common_channels = set(channel_sets[0])

    for channel_set in channel_sets[1:]:
        common_channels &= channel_set

    return common_channels


tbi_common_channels = intersect_channel_sets(tbi_channel_sets)
control_common_channels = intersect_channel_sets(control_channel_sets)

all_common_channels = tbi_common_channels & control_common_channels

# Для воспроизводимости сортируем каналы.
COMMON_CHANNELS = sorted(all_common_channels)

common_channels_path = OUT["tables"] / "common_channels.txt"

with open(common_channels_path, "w", encoding="utf-8") as file:
    for channel_name in COMMON_CHANNELS:
        file.write(f"{channel_name}\n")

print("Общий набор каналов")
print("-" * 60)
print(f"Каналов, общих для всех записей ЧМТ: {len(tbi_common_channels)}")
print(
    "Каналов, общих для всех записей контроля: "
    f"{len(control_common_channels)}"
)
print(f"Каналов, общих для обеих групп: {len(COMMON_CHANNELS)}")
print(f"Список сохранён: {common_channels_path}")

print("\nОбщие каналы:")
print(COMMON_CHANNELS)

In [ ]:
# @title 6.3. Сводка доступности каналов по группам

tbi_all_channels = sorted(set().union(*tbi_channel_sets))
control_all_channels = sorted(set().union(*control_channel_sets))

all_observed_channels = sorted(
    set(tbi_all_channels) | set(control_all_channels)
)

channel_availability = pd.DataFrame(
    {
        "channel": all_observed_channels,
    }
)

channel_availability["in_tbi"] = channel_availability["channel"].isin(
    tbi_all_channels
)

channel_availability["in_control"] = channel_availability["channel"].isin(
    control_all_channels
)

channel_availability["in_common_set"] = channel_availability["channel"].isin(
    COMMON_CHANNELS
)

channel_availability_path = OUT["tables"] / "channel_availability.csv"
channel_availability.to_csv(channel_availability_path, index=False)

print("Сводка доступности каналов:")
display(channel_availability)

logger.info(
    "Сводка доступности каналов сохранена: %s",
    channel_availability_path,
)

In [ ]:
# @title 6.3.1. Каналы, исключённые при унификации

excluded_channels = channel_availability[
    ~channel_availability["in_common_set"]
].copy()

excluded_channels["status"] = np.select(
    [
        excluded_channels["in_tbi"] & ~excluded_channels["in_control"],
        ~excluded_channels["in_tbi"] & excluded_channels["in_control"],
    ],
    [
        "есть только в ЧМТ",
        "есть только в контроле",
    ],
    default="нет в обеих группах",
)

print("Каналы, не вошедшие в общий набор:")
display(
    excluded_channels[
        ["channel", "in_tbi", "in_control", "status"]
    ].sort_values(["status", "channel"])
)

excluded_channels_path = OUT["tables"] / "excluded_channels_after_intersection.csv"
excluded_channels.to_csv(excluded_channels_path, index=False)

print(f"Таблица сохранена: {excluded_channels_path}")

## 6.4. Применение общего набора каналов

После определения общего набора каналов все стандартизированные записи приводятся к этому набору.

Для каждой записи:

1. загружается сохранённый `.fif`-файл с эпохами;
2. выбираются только каналы из `COMMON_CHANNELS`;
3. порядок каналов приводится к единому;
4. запись сохраняется в отдельную папку `epochs_common_channels`.

Исходные стандартизированные файлы не перезаписываются.

In [ ]:
# @title 6.4. Применение общего набора каналов к сохранённым эпохам

COMMON_EPOCHS_DIR = OUTPUT_DIR / "epochs_common_channels"
COMMON_EPOCHS_DIR.mkdir(parents=True, exist_ok=True)


def apply_common_channels_to_epochs_file(
    epochs_path: Path,
    common_channels: list[str],
    output_dir: Path,
    group: str,
    record_id: str,
) -> str:
    """
    Загружает Epochs, оставляет общий набор каналов и сохраняет файл.

    Parameters
    ----------
    epochs_path : Path
        Путь к исходному .fif-файлу эпох.
    common_channels : list[str]
        Единый список каналов.
    output_dir : Path
        Папка для сохранения обновлённых эпох.
    group : str
        Группа данных.
    record_id : str
        Идентификатор записи.

    Returns
    -------
    str
        Путь к сохранённому файлу.
    """
    epochs = mne.read_epochs(
        epochs_path,
        preload=True,
        verbose=False,
    )

    # Сохраняем только общий набор каналов в едином порядке.
    epochs.pick(common_channels)

    group_output_dir = output_dir / group.lower()
    group_output_dir.mkdir(parents=True, exist_ok=True)

    safe_record_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(record_id))
    output_path = group_output_dir / f"{safe_record_id}_common_epo.fif"

    epochs.save(
        output_path,
        overwrite=True,
        verbose=False,
    )

    return str(output_path)


common_channel_records = []

for _, row in tqdm(
    standardized_inventory.iterrows(),
    total=len(standardized_inventory),
    desc="Унификация каналов",
):
    epochs_path = row.get("epochs_path")

    if pd.isna(epochs_path) or epochs_path is None:
        register_processing_error(
            group=row["group"],
            file_path=Path(row["source_path"]),
            stage="common_channels",
            error=FileNotFoundError("Не указан путь к сохранённым эпохам."),
        )
        continue

    try:
        common_epochs_path = apply_common_channels_to_epochs_file(
            epochs_path=Path(epochs_path),
            common_channels=COMMON_CHANNELS,
            output_dir=COMMON_EPOCHS_DIR,
            group=row["group"],
            record_id=row["record_id"],
        )

        updated_row = row.to_dict()
        updated_row["common_epochs_path"] = common_epochs_path
        updated_row["n_common_channels"] = len(COMMON_CHANNELS)
        updated_row["common_channels"] = "|".join(COMMON_CHANNELS)

        common_channel_records.append(updated_row)

    except Exception as error:
        register_processing_error(
            group=row["group"],
            file_path=Path(row["source_path"]),
            stage="common_channels",
            error=error,
        )

common_channels_inventory = pd.DataFrame(common_channel_records)

common_channels_inventory_path = (
    OUT["tables"] / "inventory_epochs_common_channels.csv"
)

common_channels_inventory.to_csv(
    common_channels_inventory_path,
    index=False,
)

print("Унификация каналов завершена.")
print(f"Успешно обработано записей: {len(common_channels_inventory)}")
print(f"Число общих каналов: {len(COMMON_CHANNELS)}")
print(f"Файл сохранён: {common_channels_inventory_path}")

display(common_channels_inventory.head())

## 6.5. Проверка результата унификации каналов

После применения общего набора каналов проверяется, что все записи имеют одинаковое число каналов и одинаковый порядок каналов.

Это необходимо перед расчётом признаков, особенно для ROI-агрегации и сетевого анализа.

In [ ]:
# @title 6.5. Проверка результата унификации каналов

print("Проверка результата унификации каналов")
print("-" * 60)

print(f"Число записей после унификации: {len(common_channels_inventory)}")

print("\nЧисло записей по группам:")
display(
    common_channels_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
        median_epochs_per_record=("n_epochs", "median"),
        n_common_channels=("n_common_channels", "median"),
    )
    .reset_index()
)

print("\nРаспределение числа общих каналов:")
display(
    common_channels_inventory["n_common_channels"]
    .value_counts()
    .sort_index()
)

if common_channels_inventory["n_common_channels"].nunique() == 1:
    print("\nВсе записи имеют одинаковое число каналов.")
else:
    logger.warning("В записях обнаружено разное число каналов после унификации.")

## 6.6. Обзорная визуализация качества и объёма данных

После стандартизации и унификации каналов строится обзорная QC-визуализация.

Она показывает:

- распределение числа эпох на запись;
- распределение размеров исходных файлов;
- проверку длительности эпох;
- проверку числа каналов после стандартизации.

Цель рисунка — подтвердить техническую сопоставимость данных перед переходом к расчёту признаков.

In [ ]:
# @title 6.6. Обзорная сводка качества и объёма данных

def format_median_iqr(values: pd.Series) -> str:
    """
    Формирует строку с медианой и межквартильным размахом.

    Parameters
    ----------
    values : pd.Series
        Числовой ряд.

    Returns
    -------
    str
        Строка формата median, IQR.
    """
    median = values.median()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)

    return f"median={median:.2f}, IQR=[{q1:.2f}; {q3:.2f}]"


plot_inventory = common_channels_inventory.copy()

# Если file_size_mb не попал в common_channels_inventory,
# подтягиваем его из исходных inventory-таблиц.
if "file_size_mb" not in plot_inventory.columns:
    tbi_sizes = tbi_inventory[
        ["record_id", "file_size_mb"]
    ].copy()

    control_sizes = control_selected.copy()
    control_sizes["record_id"] = control_sizes["file_name"].str.replace(
        ".edf",
        "",
        regex=False,
    )

    control_sizes["file_size_mb"] = control_sizes["file_path"].apply(
        lambda path: Path(path).stat().st_size / (1024 ** 2)
    )

    size_table = pd.concat(
        [
            tbi_sizes[["record_id", "file_size_mb"]],
            control_sizes[["record_id", "file_size_mb"]],
        ],
        ignore_index=True,
    )

    plot_inventory = plot_inventory.merge(
        size_table,
        on="record_id",
        how="left",
    )

tbi_data = plot_inventory[plot_inventory["group"] == "TBI"]
control_data = plot_inventory[plot_inventory["group"] == "Control"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Распределение числа эпох.
axes[0, 0].hist(
    tbi_data["n_epochs"],
    bins=20,
    density=True,
    color=COL_TBI,
    label=f"ЧМТ (n={len(tbi_data)})",
)
axes[0, 0].hist(
    control_data["n_epochs"],
    bins=20,
    density=True,
    color=COL_CONTROL,
    label=f"Контроль (n={len(control_data)})",
)
axes[0, 0].set_title("Распределение числа эпох на запись")
axes[0, 0].set_xlabel("Число эпох по 4 секунды")
axes[0, 0].set_ylabel("Плотность")

text_epochs = (
    f"ЧМТ: {format_median_iqr(tbi_data['n_epochs'])}\n"
    f"Контроль: {format_median_iqr(control_data['n_epochs'])}"
)
axes[0, 0].text(
    0.98,
    0.95,
    text_epochs,
    transform=axes[0, 0].transAxes,
    ha="right",
    va="top",
)

# 2. Распределение размеров файлов.
axes[0, 1].hist(
    tbi_data["file_size_mb"],
    bins=20,
    density=True,
    color=COL_TBI,
    label=f"ЧМТ (n={len(tbi_data)})",
)
axes[0, 1].hist(
    control_data["file_size_mb"],
    bins=20,
    density=True,
    color=COL_CONTROL,
    label=f"Контроль (n={len(control_data)})",
)
axes[0, 1].set_title("Распределение размеров файлов")
axes[0, 1].set_xlabel("Размер файла, МБ")
axes[0, 1].set_ylabel("Плотность")

text_sizes = (
    f"ЧМТ: {format_median_iqr(tbi_data['file_size_mb'])}\n"
    f"Контроль: {format_median_iqr(control_data['file_size_mb'])}"
)
axes[0, 1].text(
    0.98,
    0.95,
    text_sizes,
    transform=axes[0, 1].transAxes,
    ha="right",
    va="top",
)

# 3. Проверка длительности эпох.
epoch_len_counts = (
    plot_inventory
    .assign(epoch_len_rounded=plot_inventory["epoch_len_sec"].round(3))
    .groupby(["group", "epoch_len_rounded"])
    .size()
    .reset_index(name="n_records")
)

for group_name, color in GROUP_COLORS.items():
    group_counts = epoch_len_counts[
        epoch_len_counts["group"] == group_name
    ]

    axes[1, 0].bar(
        group_counts["epoch_len_rounded"].astype(str),
        group_counts["n_records"],
        color=color,
        label=GROUP_LABELS_RU[group_name],
    )

axes[1, 0].axvline(
    x=0.5,
    color="black",
    linestyle="--",
    linewidth=1.5,
)
axes[1, 0].set_title("Проверка длительности эпох")
axes[1, 0].set_xlabel("Длительность эпохи, сек")
axes[1, 0].set_ylabel("Количество записей")

text_epoch_len = (
    f"ЧМТ: {len(tbi_data)}/{len(tbi_data)} записей ~ 4.0 с\n"
    f"Контроль: {len(control_data)}/{len(control_data)} записей ~ 4.0 с"
)
axes[1, 0].text(
    0.98,
    0.95,
    text_epoch_len,
    transform=axes[1, 0].transAxes,
    ha="right",
    va="top",
)

# 4. Проверка числа каналов.
channel_counts = (
    plot_inventory
    .groupby(["group", "n_common_channels"])
    .size()
    .reset_index(name="n_records")
)

for group_name, color in GROUP_COLORS.items():
    group_counts = channel_counts[
        channel_counts["group"] == group_name
    ]

    axes[1, 1].bar(
        group_counts["n_common_channels"].astype(str),
        group_counts["n_records"],
        color=color,
        label=GROUP_LABELS_RU[group_name],
    )

axes[1, 1].axvline(
    x=0.5,
    color="black",
    linestyle="--",
    linewidth=1.5,
)
axes[1, 1].set_title("Проверка числа каналов после стандартизации")
axes[1, 1].set_xlabel("Число каналов")
axes[1, 1].set_ylabel("Количество записей")

# Общая легенда снизу.
handles = [
    mpatches.Patch(
        color=COL_TBI,
        label=f"ЧМТ (n={len(tbi_data)})",
    ),
    mpatches.Patch(
        color=COL_CONTROL,
        label=f"Контроль (n={len(control_data)})",
    ),
]

fig.legend(
    handles=handles,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, -0.02),
)

fig.suptitle(
    "Обзорная сводка качества и объёма данных",
    fontsize=18,
    y=1.02,
)
fig.tight_layout()

figure_path = OUT["qc"] / "qc_overview_data_volume.png"
save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 6.7. Карточки технической сводки данных

summary_cards = (
    plot_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs_total=("n_epochs", "sum"),
        median_epochs=("n_epochs", "median"),
        median_channels=("n_common_channels", "median"),
        median_epoch_len=("epoch_len_sec", "median"),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.axis("off")

card_texts = []

for _, row in summary_cards.iterrows():
    group_label = GROUP_LABELS_RU[row["group"]]

    text = (
        f"{group_label}\n"
        f"Субъекты: {int(row['n_subjects'])}\n"
        f"Записи: {int(row['n_records'])}\n"
        f"Эпохи: {int(row['n_epochs_total'])}\n"
        f"Медиана эпох/запись: {row['median_epochs']:.0f}\n"
        f"Каналы: {row['median_channels']:.0f}\n"
        f"Длительность эпохи: {row['median_epoch_len']:.3f} с"
    )

    card_texts.append((row["group"], text))

x_positions = [0.25, 0.75]

for x_pos, (group_name, text) in zip(x_positions, card_texts):
    ax.text(
        x_pos,
        0.5,
        text,
        ha="center",
        va="center",
        fontsize=13,
        color="white",
        bbox={
            "boxstyle": "round,pad=0.8",
            "facecolor": GROUP_COLORS[group_name],
            "edgecolor": "none",
        },
        transform=ax.transAxes,
    )

ax.set_title(
    "Техническая сводка после стандартизации",
    fontsize=16,
    pad=20,
)

figure_path = OUT["qc"] / "qc_summary_cards.png"
save_figure(fig, figure_path)

plt.show()

# 7. Контроль качества стандартизированных эпох

После унификации каналов необходимо выполнить базовый контроль качества записей и эпох.

На этом этапе проверяются технические характеристики данных:

- число успешно обработанных записей;
- число эпох в каждой записи;
- число каналов;
- частота дискретизации;
- длительность эпох;
- распределение амплитуд;
- наличие записей с аномально малым числом эпох;
- наличие потенциально артефактных записей.

Контроль качества нужен для того, чтобы исключить технически некорректные или явно нестабильные записи до расчёта признаков.

In [ ]:
# @title 7.1. Общая сводка после унификации каналов

qc_summary = (
    common_channels_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs_total=("n_epochs", "sum"),
        median_epochs_per_record=("n_epochs", "median"),
        min_epochs_per_record=("n_epochs", "min"),
        max_epochs_per_record=("n_epochs", "max"),
        median_channels=("n_common_channels", "median"),
        median_sfreq=("sfreq_hz", "median"),
        median_epoch_len_sec=("epoch_len_sec", "median"),
    )
    .reset_index()
)

qc_summary_path = OUT["tables"] / "qc_summary_after_common_channels.csv"
qc_summary.to_csv(qc_summary_path, index=False)

print("Общая сводка после унификации каналов:")
display(qc_summary)

logger.info("QC-сводка сохранена: %s", qc_summary_path)

In [ ]:
# @title 7.1.1. Распределение числа эпох на запись

fig, ax = plt.subplots(figsize=(10, 6))

for group_name, color in GROUP_COLORS.items():
    group_data = common_channels_inventory[
        common_channels_inventory["group"] == group_name
    ]

    ax.hist(
        group_data["n_epochs"],
        bins=25,
        density=True,
        color=color,
        label=(
            f"{GROUP_LABELS_RU[group_name]} "
            f"(n={len(group_data)})"
        ),
    )

ax.set_title("Распределение числа 4-секундных эпох на запись")
ax.set_xlabel("Число эпох")
ax.set_ylabel("Плотность")
ax.legend(frameon=False)

figure_path = OUT["qc"] / "n_epochs_distribution.png"
save_figure(fig, figure_path)

plt.show()

## 7.2. Проверка записей с малым числом эпох

Записи с очень малым числом эпох могут давать нестабильные оценки признаков.

На этом этапе выделяются записи, в которых число 4-секундных эпох меньше заданного минимального порога.

Такие записи не обязательно автоматически исключаются, но должны быть отмечены в таблице контроля качества.

In [ ]:
# @title 7.2. Поиск записей с малым числом эпох

MIN_EPOCHS_PER_RECORD = 10

low_epoch_records = common_channels_inventory[
    common_channels_inventory["n_epochs"] < MIN_EPOCHS_PER_RECORD
].copy()

low_epoch_records_path = OUT["qc"] / "low_epoch_records.csv"
low_epoch_records.to_csv(low_epoch_records_path, index=False)

print("Проверка записей с малым числом эпох")
print("-" * 60)
print(f"Минимальный порог эпох: {MIN_EPOCHS_PER_RECORD}")
print(f"Найдено записей ниже порога: {len(low_epoch_records)}")

if len(low_epoch_records) > 0:
    display(
        low_epoch_records[
            [
                "group",
                "subject_id",
                "record_id",
                "n_epochs",
                "source_path",
            ]
        ]
    )
else:
    print("Записей с малым числом эпох не обнаружено.")

logger.info(
    "Таблица записей с малым числом эпох сохранена: %s",
    low_epoch_records_path,
)

## 7.3. Амплитудный контроль качества эпох

Для оценки выраженных артефактов рассчитывается peak-to-peak амплитуда эпох.

Для каждой записи оцениваются:

- медианная peak-to-peak амплитуда;
- 95-й процентиль peak-to-peak амплитуды;
- максимальная peak-to-peak амплитуда;
- доля эпох, превышающих заданный порог.

Порог используется как индикатор потенциально артефактных фрагментов. Он не заменяет полноценную очистку артефактов, но позволяет выявить записи с выраженными техническими проблемами.

In [ ]:
# @title 7.3. Расчёт амплитудных QC-метрик

def compute_epoch_amplitude_qc(
    epochs_path: Path,
    p2p_threshold_uv: float,
) -> dict:
    """
    Рассчитывает амплитудные QC-метрики для файла эпох.

    Parameters
    ----------
    epochs_path : Path
        Путь к .fif-файлу с эпохами.
    p2p_threshold_uv : float
        Порог peak-to-peak амплитуды в микровольтах.

    Returns
    -------
    dict
        Словарь с QC-метриками.
    """
    epochs = mne.read_epochs(
        epochs_path,
        preload=True,
        verbose=False,
    )

    data_uv = epochs.get_data(copy=True) * 1e6

    # Peak-to-peak амплитуда считается по каждой эпохе и каналу.
    p2p_by_epoch_channel = np.ptp(data_uv, axis=2)

    # Для каждой эпохи берём максимум по каналам.
    p2p_by_epoch = np.max(p2p_by_epoch_channel, axis=1)

    bad_epoch_mask = p2p_by_epoch > p2p_threshold_uv

    return {
        "p2p_median_uv": float(np.median(p2p_by_epoch)),
        "p2p_p95_uv": float(np.percentile(p2p_by_epoch, 95)),
        "p2p_max_uv": float(np.max(p2p_by_epoch)),
        "n_bad_epochs_p2p": int(np.sum(bad_epoch_mask)),
        "prop_bad_epochs_p2p": float(np.mean(bad_epoch_mask)),
    }


amplitude_qc_records = []

for _, row in tqdm(
    common_channels_inventory.iterrows(),
    total=len(common_channels_inventory),
    desc="Амплитудный QC",
):
    try:
        qc_metrics = compute_epoch_amplitude_qc(
            epochs_path=Path(row["common_epochs_path"]),
            p2p_threshold_uv=CONFIG["drop_epoch_p2p_uv"],
        )

        record = {
            "group": row["group"],
            "subject_id": row["subject_id"],
            "record_id": row["record_id"],
            "n_epochs": row["n_epochs"],
            "common_epochs_path": row["common_epochs_path"],
        }
        record.update(qc_metrics)

        amplitude_qc_records.append(record)

    except Exception as error:
        register_processing_error(
            group=row["group"],
            file_path=Path(row["source_path"]),
            stage="amplitude_qc",
            error=error,
        )

amplitude_qc = pd.DataFrame(amplitude_qc_records)

amplitude_qc_path = OUT["qc"] / "amplitude_qc_by_record.csv"
amplitude_qc.to_csv(amplitude_qc_path, index=False)

print("Амплитудный QC завершён.")
print(f"Записей с QC-метриками: {len(amplitude_qc)}")
print(f"Файл сохранён: {amplitude_qc_path}")

display(amplitude_qc.head())

In [ ]:
# @title 7.4. Сводка амплитудного QC по группам

amplitude_qc_summary = (
    amplitude_qc
    .groupby("group")
    .agg(
        n_records=("record_id", "count"),
        median_p2p_uv=("p2p_median_uv", "median"),
        median_p95_p2p_uv=("p2p_p95_uv", "median"),
        median_prop_bad_epochs=("prop_bad_epochs_p2p", "median"),
        max_prop_bad_epochs=("prop_bad_epochs_p2p", "max"),
    )
    .reset_index()
)

amplitude_qc_summary_path = OUT["qc"] / "amplitude_qc_summary_by_group.csv"
amplitude_qc_summary.to_csv(amplitude_qc_summary_path, index=False)

print("Сводка амплитудного QC по группам:")
display(amplitude_qc_summary)

logger.info(
    "Сводка амплитудного QC сохранена: %s",
    amplitude_qc_summary_path,
)

In [ ]:
# @title 7.4.1. Распределение доли эпох выше p2p-порога

fig, ax = plt.subplots(figsize=(10, 6))

for group_name, color in GROUP_COLORS.items():
    group_data = amplitude_qc[
        amplitude_qc["group"] == group_name
    ]

    ax.hist(
        group_data["prop_bad_epochs_p2p"],
        bins=25,
        density=True,
        color=color,
        label=(
            f"{GROUP_LABELS_RU[group_name]} "
            f"(n={len(group_data)})"
        ),
    )

ax.axvline(
    CONFIG["max_bad_epoch_prop"],
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Порог исключения",
)

ax.set_title("Доля эпох выше p2p-порога")
ax.set_xlabel("Доля эпох выше порога")
ax.set_ylabel("Плотность")
ax.legend(frameon=False)

figure_path = OUT["qc"] / "prop_bad_epochs_p2p_distribution.png"
save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 7.4.2. Распределение медианной p2p-амплитуды по записям

fig, ax = plt.subplots(figsize=(10, 6))

for group_name, color in GROUP_COLORS.items():
    group_data = amplitude_qc[
        amplitude_qc["group"] == group_name
    ]

    ax.hist(
        group_data["p2p_median_uv"],
        bins=25,
        density=True,
        color=color,
        label=(
            f"{GROUP_LABELS_RU[group_name]} "
            f"(n={len(group_data)})"
        ),
    )

ax.axvline(
    CONFIG["drop_epoch_p2p_uv"],
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="p2p-порог",
)

ax.set_title("Распределение медианной peak-to-peak амплитуды")
ax.set_xlabel("Медианная p2p-амплитуда, мкВ")
ax.set_ylabel("Плотность")
ax.legend(frameon=False)

figure_path = OUT["qc"] / "median_p2p_amplitude_distribution.png"
save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 7.4.3. Поиск записей с высокой долей артефактных эпох

BAD_EPOCH_PROP_WARNING = 0.30
BAD_EPOCH_PROP_EXCLUDE = CONFIG["max_bad_epoch_prop"]

high_artifact_records = amplitude_qc[
    amplitude_qc["prop_bad_epochs_p2p"] >= BAD_EPOCH_PROP_WARNING
].copy()

high_artifact_records = high_artifact_records.sort_values(
    ["prop_bad_epochs_p2p", "p2p_median_uv"],
    ascending=[False, False],
)

high_artifact_records_path = (
    OUT["qc"] / "high_artifact_records_amplitude_qc.csv"
)
high_artifact_records.to_csv(high_artifact_records_path, index=False)

print("Записи с высокой долей эпох выше p2p-порога")
print("-" * 70)
print(f"Порог предупреждения: {BAD_EPOCH_PROP_WARNING:.2f}")
print(f"Порог исключения: {BAD_EPOCH_PROP_EXCLUDE:.2f}")
print(f"Найдено записей: {len(high_artifact_records)}")
print(f"Файл сохранён: {high_artifact_records_path}")

display(
    high_artifact_records[
        [
            "group",
            "subject_id",
            "record_id",
            "n_epochs",
            "p2p_median_uv",
            "p2p_p95_uv",
            "p2p_max_uv",
            "n_bad_epochs_p2p",
            "prop_bad_epochs_p2p",
        ]
    ].head(30)
)

In [ ]:
# @title 7.4.4. Распределение медианной p2p-амплитуды без экстремальных выбросов

P2P_XLIM_UV = 1000

fig, ax = plt.subplots(figsize=(11, 6))

for group_name, color in GROUP_COLORS.items():
    group_data = amplitude_qc[
        amplitude_qc["group"] == group_name
    ]

    ax.hist(
        group_data["p2p_median_uv"],
        bins=30,
        density=True,
        color=color,
        label=(
            f"{GROUP_LABELS_RU[group_name]} "
            f"(n={len(group_data)})"
        ),
    )

ax.axvline(
    CONFIG["drop_epoch_p2p_uv"],
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="p2p-порог",
)

ax.set_xlim(0, P2P_XLIM_UV)

ax.set_title(
    "Распределение медианной peak-to-peak амплитуды\n"
    "масштабировано до 1000 мкВ"
)
ax.set_xlabel("Медианная p2p-амплитуда, мкВ")
ax.set_ylabel("Плотность")
ax.legend(frameon=False)

figure_path = OUT["qc"] / "median_p2p_amplitude_distribution_zoom.png"
save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 7.4.5. Доля эпох выше p2p-порога по записям

fig, ax = plt.subplots(figsize=(11, 6))

for group_name, color in GROUP_COLORS.items():
    group_data = amplitude_qc[
        amplitude_qc["group"] == group_name
    ]

    ax.hist(
        group_data["prop_bad_epochs_p2p"],
        bins=np.linspace(0, 1, 31),
        density=False,
        color=color,
        label=(
            f"{GROUP_LABELS_RU[group_name]} "
            f"(n={len(group_data)})"
        ),
    )

ax.axvline(
    CONFIG["max_bad_epoch_prop"],
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Порог исключения",
)

ax.set_title("Доля эпох, превышающих p2p-порог")
ax.set_xlabel("Доля эпох выше порога")
ax.set_ylabel("Количество записей")
ax.legend(frameon=False)

figure_path = OUT["qc"] / "prop_bad_epochs_p2p_counts.png"
save_figure(fig, figure_path)

plt.show()

## 7.5. Формирование таблицы пригодных записей

На основе технических и амплитудных QC-метрик формируется таблица записей, пригодных для дальнейшего анализа.

Запись исключается, если:

- число эпох меньше минимального порога;
- доля эпох с peak-to-peak амплитудой выше порога превышает допустимое значение.

Итоговая таблица используется как вход для расчёта признаков.

In [ ]:
# @title 7.5. Формирование итоговой таблицы пригодных записей

analysis_inventory = common_channels_inventory.merge(
    amplitude_qc,
    on=[
        "group",
        "subject_id",
        "record_id",
        "n_epochs",
        "common_epochs_path",
    ],
    how="left",
)

analysis_inventory["qc_pass_n_epochs"] = (
    analysis_inventory["n_epochs"] >= CONFIG["min_epochs_per_record"]
)

analysis_inventory["qc_pass_amplitude"] = (
    analysis_inventory["prop_bad_epochs_p2p"]
    <= CONFIG["max_bad_epoch_prop"]
)

analysis_inventory["qc_pass"] = (
    analysis_inventory["qc_pass_n_epochs"]
    & analysis_inventory["qc_pass_amplitude"]
)

analysis_inventory_path = OUT["qc"] / "analysis_inventory_with_qc.csv"
analysis_inventory.to_csv(analysis_inventory_path, index=False)

analysis_ready_inventory = analysis_inventory[
    analysis_inventory["qc_pass"]
].copy()

analysis_ready_inventory_path = OUT["tables"] / "analysis_ready_inventory.csv"
analysis_ready_inventory.to_csv(analysis_ready_inventory_path, index=False)

print("Формирование таблицы пригодных записей завершено.")
print(f"Всего записей: {len(analysis_inventory)}")
print(f"Прошли QC: {len(analysis_ready_inventory)}")
print(f"Исключены: {len(analysis_inventory) - len(analysis_ready_inventory)}")

print("\nСводка QC по группам:")
display(
    analysis_inventory
    .groupby(["group", "qc_pass"])
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
    )
    .reset_index()
)

print(f"\nФайл с QC сохранён: {analysis_inventory_path}")
print(f"Файл для анализа сохранён: {analysis_ready_inventory_path}")

In [ ]:
# @title 7.6. Сохранение таблицы ошибок обработки

processing_errors_path = OUT["errors"] / "processing_errors.csv"

save_processing_errors(
    errors=processing_errors,
    output_path=processing_errors_path,
)

print("Проверка ошибок обработки завершена.")
print(f"Число зарегистрированных ошибок: {len(processing_errors)}")

if processing_errors:
    display(pd.DataFrame(processing_errors).head())

In [ ]:
# @title 7.5.1. Визуальная сводка результата QC-отбора

qc_pass_summary = (
    analysis_inventory
    .groupby(["group", "qc_pass"])
    .agg(n_records=("record_id", "count"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 6))

x_labels = []
bar_values = []
bar_colors = []

for group_name in ["TBI", "Control"]:
    for qc_status in [True, False]:
        subset = qc_pass_summary[
            (qc_pass_summary["group"] == group_name)
            & (qc_pass_summary["qc_pass"] == qc_status)
        ]

        value = int(subset["n_records"].iloc[0]) if len(subset) > 0 else 0
        status_label = "прошли QC" if qc_status else "исключены"

        x_labels.append(f"{GROUP_LABELS_RU[group_name]}\n{status_label}")
        bar_values.append(value)
        bar_colors.append(GROUP_COLORS[group_name])

bars = ax.bar(
    x_labels,
    bar_values,
    color=bar_colors,
)

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 2,
        str(int(height)),
        ha="center",
        va="bottom",
    )

ax.set_title("Результат базового QC-отбора записей")
ax.set_ylabel("Количество записей")

figure_path = OUT["qc"] / "qc_pass_summary.png"
save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 7.5.2. Итоговая сводка выборки после QC

final_qc_summary = (
    analysis_ready_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
        median_epochs_per_record=("n_epochs", "median"),
        min_epochs_per_record=("n_epochs", "min"),
        max_epochs_per_record=("n_epochs", "max"),
        n_common_channels=("n_common_channels", "median"),
        median_sfreq=("sfreq_hz", "median"),
        median_epoch_len_sec=("epoch_len_sec", "median"),
    )
    .reset_index()
)

final_qc_summary["group_ru"] = final_qc_summary["group"].map(GROUP_LABELS_RU)

final_qc_summary_path = OUT["tables"] / "final_qc_summary.csv"
final_qc_summary.to_csv(final_qc_summary_path, index=False)

print("Итоговая сводка выборки после QC:")
display(final_qc_summary)

print(f"Файл сохранён: {final_qc_summary_path}")

# 8. Итог подготовки данных

На данном этапе завершена подготовка ЭЭГ-данных для последующего количественного анализа.

В рамках ноутбука были выполнены следующие операции:

1. проверены локальные пути к исходным данным;
2. сформированы inventory-таблицы для группы ЧМТ и контрольной группы;
3. выбраны контрольные записи OpenNeuro ds005385;
4. выполнена загрузка данных из `.mat` и EDF-файлов;
5. данные приведены к формату MNE Epochs;
6. выполнены ресемплинг, фильтрация и референсирование;
7. определён общий набор из 58 ЭЭГ-каналов;
8. все записи приведены к единому пространству каналов;
9. выполнен базовый амплитудный контроль качества;
10. сформирована итоговая таблица записей, пригодных для анализа.

После контроля качества в анализ включены:

| Группа | Субъекты | Записи | Эпохи | Каналы |
|---|---:|---:|---:|---:|
| ЧМТ | 89 | 199 | 22 863 | 58 |
| Контроль | 72 | 356 | 16 470 | 58 |

Основным результатом ноутбука является файл:

```text
analysis_ready_inventory.csv

In [ ]:

# @title 8.1. Финальная проверка ключевых выходных файлов

required_outputs = {
    "analysis_ready_inventory": OUT["tables"] / "analysis_ready_inventory.csv",
    "final_qc_summary": OUT["tables"] / "final_qc_summary.csv",
    "standardized_inventory": OUT["tables"] / "inventory_epochs_standardized.csv",
    "common_channels_inventory": (
        OUT["tables"] / "inventory_epochs_common_channels.csv"
    ),
    "common_channels": OUT["tables"] / "common_channels.txt",
    "analysis_inventory_with_qc": (
        OUT["qc"] / "analysis_inventory_with_qc.csv"
    ),
    "excluded_records_after_qc": (
        OUT["qc"] / "excluded_records_after_qc.csv"
    ),
    "processing_errors": OUT["errors"] / "processing_errors.csv",
}

print("Проверка ключевых выходных файлов")
print("-" * 70)

for name, path in required_outputs.items():
    exists = path.exists()
    status = "OK" if exists else "НЕ НАЙДЕН"

    print(f"{name}: {status}")
    print(f"  {path}")

print("\nПодготовка данных завершена.")
print("Следующий этап: количественный анализ ЭЭГ.")

In [ ]:
# @title 8.2. Сохранение таблицы исключённых после QC записей

excluded_records = analysis_inventory[
    ~analysis_inventory["qc_pass"]
].copy()

excluded_records_path = OUT["qc"] / "excluded_records_after_qc.csv"
excluded_records.to_csv(excluded_records_path, index=False)

print("Таблица исключённых записей сохранена:")
print(excluded_records_path)
print(f"Исключено записей: {len(excluded_records)}")

display(
    excluded_records[
        [
            "group",
            "subject_id",
            "record_id",
            "n_epochs",
            "p2p_median_uv",
            "p2p_p95_uv",
            "p2p_max_uv",
            "prop_bad_epochs_p2p",
            "qc_pass_n_epochs",
            "qc_pass_amplitude",
            "qc_pass",
        ]
    ].sort_values("prop_bad_epochs_p2p", ascending=False)
)

In [ ]:
# @title 8.3. Сохранение таблицы ошибок обработки

processing_errors_path = OUT["errors"] / "processing_errors.csv"

error_columns = [
    "group",
    "file_path",
    "stage",
    "error_type",
    "error_message",
]

if len(processing_errors) > 0:
    processing_errors_df = pd.DataFrame(processing_errors)
else:
    processing_errors_df = pd.DataFrame(columns=error_columns)

processing_errors_df.to_csv(processing_errors_path, index=False)

print("Таблица ошибок обработки сохранена:")
print(processing_errors_path)
print(f"Число зарегистрированных ошибок: {len(processing_errors_df)}")

display(processing_errors_df.head())

### Финальная проверка выходных файлов

Финальная проверка подтвердила, что все ключевые результаты этапа предобработки сохранены.

Основные выходные файлы включают:

- итоговую таблицу записей, пригодных для анализа;
- сводку после QC;
- объединённый inventory стандартизированных эпох;
- inventory после унификации каналов;
- список 58 общих каналов;
- полную QC-таблицу;
- таблицу исключённых записей;
- таблицу ошибок обработки.

Эти файлы фиксируют состав данных после предобработки и позволяют воспроизвести переход от исходных локальных файлов к набору, пригодному для количественного анализа ЭЭГ.